In [1]:
import heapq
from collections import defaultdict, deque

# 1) Grid Traversal

**The gist.** A grid is a graph: each cell is a node, its 4 neighbours are its edges. "Islands", "regions" and "spreading" all mean: visit every cell you can reach from a start, and mark cells so you never visit one twice.

> **Think of it like…** spilling paint on a tiled floor: it spreads to every touching tile of the same kind and stops at walls. Counting islands = counting how many separate spills it takes to cover all the land.

Every graph question is two steps: **model it** (what's a node, what's an edge, weighted? directed?) and **pick the traversal**:

| Need | Algorithm | Cost |
|---|---|---|
| reachability, components, flood fill | DFS or BFS | O(V + E) |
| fewest steps, unweighted | BFS (multi-source if several starts) | O(V + E) |
| order with prerequisites, directed cycle | topological sort (Kahn) | O(V + E) |
| merging groups, undirected cycle | union–find | ~O(1) per op |
| cheapest path, non-negative weights | Dijkstra | O(E log V) |
| cheapest path with ≤ k edges | Bellman–Ford (k rounds) | O(k · E) |
| connect everything cheaply | Prim / Kruskal (MST) | O(E log V) |

```python
def dfs(r, c):
    if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != target:
        return
    grid[r][c] = visited_mark          # mark before recursing
    for dr, dc in DIRS:
        dfs(r + dr, c + dc)
```

**Spot it:** "island", "region", "connected", "flood fill", "surrounded" · "minimum minutes / steps" on a grid → BFS · "distance to the nearest X" from many X's → multi-source BFS · "water flows to the ocean" → start from the ocean instead.

**Classic bugs:** marking visited when you *pop* instead of when you *push* (cells get queued many times) · missing the bounds check · deep recursion on a 300×300 grid (use a stack or BFS).

In [2]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

## 1.1 Connected regions (DFS)

Scan every cell; each unvisited start cell kicks off one traversal that marks its whole region. Number of starts = number of regions. DFS or BFS both work — only the marking matters.

### 1.1.1 [200. Number of Islands](https://leetcode.com/problems/number-of-islands/) — ★★★ · Medium

**Problem.** Count islands of `"1"` cells (4-directionally connected) in a grid.  
**Example.** `[["1","1","0"],["1","1","0"],["0","0","1"]]` → `2` (a big island and a single cell)  
**Spot it.** Count connected components → start a traversal from every unvisited land cell.

**Ask first**
- Can I modify the grid? → If not, use a `visited` set (O(m·n) extra).
- Diagonals count? → No, 4 directions only.

> 💡 **Idea.** Every time you step on unvisited land, that's a new island — then sink the whole island so you don't count it again.

**Brute force:** union–find over all land cells — also O(m·n), just more code.

**Steps**
1. Scan every cell.
2. On unvisited land, count one island.
3. Sink the whole island (mark every reachable land cell as water).
4. Continue scanning.

**Cost:** Time O(rows · cols) (each cell is visited a constant number of times) · Space O(rows · cols) worst case for the stack

**Likely follow-ups**
- **Land is added one cell at a time; report the count after each (LC 305)?** Union–find: each new land cell adds 1 island, then union with land neighbours, −1 per successful union. O(k·α).
- **Grid is too big for memory / rows stream in?** Process row by row with union–find on just the current and previous row; carry component labels across the boundary and count components that end.
- **Count *distinct* island shapes (LC 694)?** Record each island's cells relative to its first cell (or the DFS path string) and put them in a set.
- **Recursion depth on a huge island?** Use iterative DFS with a stack or BFS.

In [3]:
def num_islands(grid):
    """Number of 4-connected land regions."""
    rows, cols = len(grid), len(grid[0])

    def sink(r, c):
        # iterative DFS; mark on push so no cell is queued twice
        stack = [(r, c)]
        grid[r][c] = "0"
        while stack:
            r, c = stack.pop()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == "1":
                    grid[nr][nc] = "0"
                    stack.append((nr, nc))

    count = 0
    for r in range(rows):
        for c in range(cols):
            # unvisited land starts a new island
            if grid[r][c] == "1":
                count += 1
                sink(r, c)
    return count


g = [list("11000"), list("11000"), list("00100"), list("00011")]
assert num_islands(g) == 3
assert num_islands([list("11110"), list("11010"), list("11000"), list("00000")]) == 1
print("ok")

ok


### 1.1.2 [695. Max Area of Island](https://leetcode.com/problems/max-area-of-island/) — ★★ · Medium

**Problem.** Area of the largest island (cells of value 1).  
**Example.** `[[1,1,0],[1,1,0],[0,0,1]]` → `4`  
**Spot it.** Components with a size → DFS returning the count.

**Ask first**
- No land at all? → Return 0.

> 💡 **Idea.** Same as counting islands, but the DFS returns how many cells it sank. Keep the biggest.

**Brute force:** a separate visited set per island — same idea, more memory.

**Steps**
1. `area(r, c)` returns 0 outside the grid or on water.
2. Mark the cell as water, return `1 + sum of neighbour areas`.
3. Take the max over all cells.

**Cost:** Time O(rows · cols) · Space O(rows · cols) recursion in the worst case

**Likely follow-ups**
- **You may flip one 0 to 1 — biggest island now (LC 827)?** Label each island with an id and store its size. For each 0, add up the sizes of the *distinct* neighbouring island ids + 1. O(n²).
- **Return the island's cells, not just the size?** Collect coordinates during the DFS for the best island.
- **Perimeter of an island (LC 463)?** For each land cell add 4, minus 2 for every land neighbour to the right or below.

In [4]:
def max_area_of_island(grid):
    """Largest island area."""
    rows, cols = len(grid), len(grid[0])

    def area(r, c):
        # stop if we walked off the map
        if not (0 <= r < rows and 0 <= c < cols):
            return 0
        # stop if it is water, or land we already counted
        if grid[r][c] != 1:
            return 0

        # drop a breadcrumb: turn land into water so we never count it twice
        grid[r][c] = 0

        # this cell, plus all the land we can walk to from it
        size = 1
        for dr, dc in DIRS:
            size += area(r + dr, c + dc)
        return size

    # try every cell as a starting point; water and counted land give 0
    biggest = 0
    for r in range(rows):
        for c in range(cols):
            biggest = max(biggest, area(r, c))
    return biggest


g = [
    [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0],
    [0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0],
]
assert max_area_of_island(g) == 6
assert max_area_of_island([[0, 0]]) == 0
print("ok")

ok


### 1.1.3 [733. Flood Fill](https://leetcode.com/problems/flood-fill/) — ★★ · Easy

**Problem.** Recolour the connected region of the start pixel's colour.  
**Example.** `image = [[1,1,1],[1,1,0],[1,0,1]], sr = 1, sc = 1, color = 2` → `[[2,2,2],[2,2,0],[2,0,1]]`  
**Spot it.** Paint-bucket = DFS from one cell.

**Ask first**
- New colour = old colour? → Return immediately, or the DFS never ends (nothing ever looks visited).

> 💡 **Idea.** Paint the start cell and spread to every 4-neighbour that still has the old colour. Painting doubles as the visited mark.

**Brute force:** a separate visited set — extra memory.

**Steps**
1. Remember the old colour; if it equals the new colour, return.
2. DFS from the start, recolouring cells that still have the old colour.
3. Recurse into the 4 neighbours.

**Cost:** Time O(rows · cols) · Space O(rows · cols) recursion worst case

In [ ]:
def flood_fill(image, sr, sc, color):
    """Image after flood fill from (sr, sc)."""

    # same colour: nothing to do (and the DFS would never stop)
    old = image[sr][sc]
    if old == color:
        return image
    
    rows, cols = len(image), len(image[0])
    def fill(r, c):
        if 0 <= r < rows and 0 <= c < cols and image[r][c] == old:
            # painting also marks the cell as visited
            image[r][c] = color
            for dr, dc in DIRS:
                fill(r + dr, c + dc)

    fill(sr, sc)
    return image


assert flood_fill([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2) == [
    [2, 2, 2],
    [2, 2, 0],
    [2, 0, 1],
]
assert flood_fill([[0, 0, 0], [0, 0, 0]], 0, 0, 0) == [[0, 0, 0], [0, 0, 0]]
print("ok")

ok


### 1.1.4 [130. Surrounded Regions](https://leetcode.com/problems/surrounded-regions/) — ★★ · Medium

**Problem.** Flip every `O` region not connected to the border into `X`.  
**Example.** `[["X","X","X"],["X","O","X"],["X","X","X"]]` → the middle `O` is surrounded, so it becomes `X`  
**Spot it.** Invert the question: find what is *safe* (border-connected) and flip the rest.

**Ask first**
- Do border `O`s count as surrounded? → No, they and everything touching them are safe.

> 💡 **Idea.** Flip it: instead of finding surrounded `O`s, find the *safe* ones — any `O` connected to the border. Mark those, then flip every other `O` to `X`.

**Brute force:** DFS from every interior `O` to see if it reaches the border, O((m·n)²).

**Steps**
1. DFS from every border `O`, marking reached cells `#`.
2. Every remaining `O` is surrounded → `X`.
3. Every `#` → back to `O`.

**Cost:** Time O(rows · cols) · Space O(rows · cols) recursion worst case

**Likely follow-ups**
- **Union–find version?** Union every border `O` with a dummy node and each `O` with its `O` neighbours; flip `O`s not connected to the dummy.
- **One huge region of `O`s?** Recursive DFS can overflow — use BFS or an explicit stack.
- **Count enclosed regions instead (LC 1254)?** Sink border-connected land first, then count the remaining islands.

In [ ]:
def solve_surrounded(board):
    """Capture surrounded regions in place."""
    rows, cols = len(board), len(board[0])

    def mark(r, c):
        # "#" = connected to the border, therefore safe
        if 0 <= r < rows and 0 <= c < cols and board[r][c] == "O":
            board[r][c] = "#"
            for dr, dc in DIRS:
                mark(r + dr, c + dc)

    # start only from border cells
    for r in range(rows):
        mark(r, 0)
        mark(r, cols - 1)

    for c in range(cols):
        mark(0, c)
        mark(rows - 1, c)
        
    # safe cells go back to O, everything else is captured
    for r in range(rows):
        for c in range(cols):
            board[r][c] = "O" if board[r][c] == "#" else "X"


b = [list("XXXX"), list("XOOX"), list("XXOX"), list("XOXX")]
solve_surrounded(b)
assert b == [list("XXXX"), list("XXXX"), list("XXXX"), list("XOXX")]
print(["".join(r) for r in b])

['XXXX', 'XXXX', 'XXXX', 'XOXX']


### 1.1.5 [417. Pacific Atlantic Water Flow](https://leetcode.com/problems/pacific-atlantic-water-flow/) — ★★★ · Medium

**Problem.** Cells from which water can flow (to equal or lower neighbours) into both the Pacific (top/left edges) and the Atlantic (bottom/right edges).  
**Example.** `[[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],[6,7,1,4,5],[5,1,1,2,4]]` → `[[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]`  
**Spot it.** Reverse the flow: climb *uphill* from each ocean's border and intersect.

**Ask first**
- Does water flow to equal heights? → Yes, `≤` — so walk uphill with `>=`.

> 💡 **Idea.** Go backwards: start from each ocean's border and walk *uphill* (to cells ≥ current). Cells reached from both oceans are the answer.

**Brute force:** from every cell, DFS downhill to see which oceans it reaches, O((m·n)²).

**Steps**
1. Start set for the Pacific: top row + left column; for the Atlantic: bottom row + right column.
2. From each start set, walk to neighbours with height ≥ the current cell.
3. Collect each ocean's reachable set.
4. Return the intersection.

**Cost:** Time O(rows · cols) · Space O(rows · cols)

**Likely follow-ups**
- **Three or more oceans?** One reverse search per ocean, then intersect all the reached sets.
- **O(1) extra space (besides the answer)?** Store reached-from-Pacific and reached-from-Atlantic as two bits inside each cell (if you may modify the grid).
- **Return cells that reach at least one ocean?** Union instead of intersection.

In [7]:
def pacific_atlantic(heights):
    """Cells that drain to both oceans."""
    rows, cols = len(heights), len(heights[0])

    def climb(starts):
        """Cells reachable from starts by moving to equal or higher cells."""
        seen = set(starts)
        stack = list(starts)
        while stack:
            r, c = stack.pop()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                # uphill (reverse of water flow)
                if (
                    0 <= nr < rows
                    and 0 <= nc < cols
                    and (nr, nc) not in seen
                    and heights[nr][nc] >= heights[r][c]
                ):
                    seen.add((nr, nc))
                    stack.append((nr, nc))
        return seen

    pacific = climb([(0, c) for c in range(cols)] + [(r, 0) for r in range(rows)])
    atlantic = climb(
        [(rows - 1, c) for c in range(cols)] + [(r, cols - 1) for r in range(rows)]
    )
    # cells reached from both oceans
    return sorted([r, c] for r, c in pacific & atlantic)


h = [
    [1, 2, 2, 3, 5],
    [3, 2, 3, 4, 4],
    [2, 4, 5, 3, 1],
    [6, 7, 1, 4, 5],
    [5, 1, 1, 2, 4],
]
assert pacific_atlantic(h) == [[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]
print(pacific_atlantic(h))

[[0, 4], [1, 3], [1, 4], [2, 2], [3, 0], [3, 1], [4, 0]]


## 1.2 Shortest paths on grids (BFS)

BFS reaches cells in order of distance, so the first time it reaches a cell is the shortest way there. **Multi-source BFS** puts every start in the queue at distance 0 — one pass gives every cell its distance to the *nearest* source. Process level by level when you need "minutes" or "steps".

### 1.2.1 [994. Rotting Oranges](https://leetcode.com/problems/rotting-oranges/) — ★★ · Medium

**Problem.** Each minute rotten oranges rot their fresh neighbours. Minutes until none are fresh (−1 if impossible)?  
**Example.** `[[2,1,1],[1,1,0],[0,1,1]]` → `4`  
**Spot it.** Simultaneous spreading from several sources → multi-source BFS by levels.

**Ask first**
- No fresh oranges at the start? → Answer 0 (not −1, not 1).
- Some fresh oranges can't be reached? → −1, so keep a `fresh` counter.

> 💡 **Idea.** All rotten oranges spread at the same time, so put them all in the queue at minute 0 and expand level by level — one level = one minute.

**By hand (think of mould spreading in a fruit bowl).** Each minute, every rotten orange infects the fresh oranges touching it (up, down, left, right). Mark the starting rotten ones, then repeat: *minute 1* infects their neighbours, *minute 2* infects the neighbours of those, and so on. Count the minutes until nothing fresh is left. If some fresh orange is walled off by empty cells, it never rots → `-1`.
In code: `queue` = oranges that rotted and still need to infect neighbours; `fresh` = how many fresh ones remain; one pass of the `while` loop = one minute.

**Brute force:** simulate minute by minute rescanning the whole grid, O((m·n)²).

**Steps**
1. Queue every rotten orange; count fresh ones.
2. While the queue is non-empty and fresh oranges remain, process one level (= one minute).
3. Each fresh neighbour turns rotten, is queued, and `fresh −= 1`.
4. Return the minutes, or −1 if fresh oranges remain.

**Cost:** Time O(rows · cols) · Space O(rows · cols)

**Likely follow-ups**
- **No separate visited array?** Turning a fresh orange into 2 *is* the visited mark.
- **Some oranges take 2 minutes to rot (different speeds)?** Edges now have different weights → Dijkstra (or 0-1 BFS / bucket queue for small weights).
- **Which orange rots last / time for each cell?** Store the minute each cell rotted in a distance grid.

In [ ]:
def oranges_rotting(grid):
    """Minutes until every orange is rotten."""
    rows, cols = len(grid), len(grid[0])

    # Step 1: look at the whole grid once to find our starting points.
    queue = deque()
    fresh = 0
    for r in range(rows):
        for c in range(cols):
            # 2 = rotten -> it starts spreading at minute 0
            if grid[r][c] == 2:
                queue.append((r, c))
            # 1 = fresh -> count it
            elif grid[r][c] == 1:
                fresh += 1
    
    
    minutes = 0
    while queue and fresh:
        minutes += 1
        # one BFS level = one minute of spreading
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    queue.append((nr, nc))
                    
    return -1 if fresh else minutes


assert oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]) == 4
assert oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]) == -1
assert oranges_rotting([[0, 2]]) == 0
print("ok")

ok


### 1.2.2 [286. Walls and Gates](https://leetcode.com/problems/walls-and-gates/) — ★★ · Medium · Premium

**Problem.** Fill each empty room (`INF`) with the distance to its nearest gate (`0`); walls are −1.  
**Example.** (`INF` = empty room) `[[INF,-1,0,INF],[INF,INF,INF,-1],[INF,-1,INF,-1],[0,-1,INF,INF]]` → `[[3,-1,0,1],[2,2,1,-1],[1,-1,2,-1],[0,-1,3,4]]`  
**Spot it.** Distance to nearest of many sources → multi-source BFS from all gates.

**Ask first**
- Update the grid in place? → Yes — rooms start as INF, which also works as the "unvisited" check.

> 💡 **Idea.** Start BFS from *all gates at once*. The first time BFS reaches a room, that's its distance to the nearest gate.

**Brute force:** BFS from each gate separately, O(gates · m · n).

**Steps**
1. Queue every gate.
2. Pop a cell; for each neighbour still `INF`, write `dist + 1` and queue it.
3. Walls and filled rooms are never touched again.

**Cost:** Time O(rows · cols) · Space O(rows · cols)

**Likely follow-ups**
- **Why multi-source instead of BFS per room or per gate?** One pass, O(m·n) total, instead of O(k·m·n).
- **Return which gate is nearest?** Carry the gate id along in the queue.
- **Weighted cells (some rooms are slower)?** Dijkstra from all gates at once.

In [ ]:
INF = 2**31 - 1

def walls_and_gates(rooms):
    """Distance to nearest gate, in place."""
    rows, cols = len(rooms), len(rooms[0])
    # all gates start together
    queue = deque((r, c) for r in range(rows) for c in range(cols) if rooms[r][c] == 0)
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            # INF = not reached yet; first arrival is the nearest gate
            if 0 <= nr < rows and 0 <= nc < cols and rooms[nr][nc] == INF:
                rooms[nr][nc] = rooms[r][c] + 1
                queue.append((nr, nc))


rooms = [[INF, -1, 0, INF], [INF, INF, INF, -1], [INF, -1, INF, -1], [0, -1, INF, INF]]
walls_and_gates(rooms)
assert rooms == [[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]
print(rooms)

[[3, -1, 0, 1], [2, 2, 1, -1], [1, -1, 2, -1], [0, -1, 3, 4]]


### 1.2.3 [542. 01 Matrix](https://leetcode.com/problems/01-matrix/) — ★★ · Medium

**Problem.** Distance of each cell to the nearest 0.  
**Example.** `[[0,0,0],[0,1,0],[1,1,1]]` → `[[0,0,0],[0,1,0],[1,2,1]]`  
**Spot it.** Multi-source BFS from every 0.

**Ask first**
- Return a new matrix or modify in place? → Either; mark 1s as unvisited (e.g. −1) first.

> 💡 **Idea.** Multi-source BFS from every 0. Each 1 gets the distance at which BFS first reaches it.

**Brute force:** BFS from every 1 to its nearest 0, O((m·n)²).

**Steps**
1. `dist` = 0 for zeros, −1 for ones.
2. Queue all zeros.
3. For each popped cell, set unvisited neighbours to `dist + 1` and queue them.

**Cost:** Time O(rows · cols) · Space O(rows · cols)

**Likely follow-ups**
- **DP without a queue?** Two passes: top-left → bottom-right using top/left neighbours, then bottom-right → top-left using bottom/right. O(m·n), O(1) extra.
- **Distance to the nearest 1 instead?** Swap the roles: start from all 1s.
- **Diagonal moves allowed?** Add the 4 diagonal directions (Chebyshev distance).

In [ ]:
def update_matrix(mat):
    """Distance from each cell to the nearest zero."""
    rows, cols = len(mat), len(mat[0])

    # -1 marks "not reached yet"
    dist = [[0 if v == 0 else -1 for v in row] for row in mat]
    queue = deque((r, c) for r in range(rows) for c in range(cols) if mat[r][c] == 0)
    
    while queue:
        r, c = queue.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                queue.append((nr, nc))
    return dist


assert update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]) == [
    [0, 0, 0],
    [0, 1, 0],
    [1, 2, 1],
]
print(update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))

[[0, 0, 0], [0, 1, 0], [1, 2, 1]]


### 1.2.4 [1091. Shortest Path in Binary Matrix](https://leetcode.com/problems/shortest-path-in-binary-matrix/) — ★ · Medium

**Problem.** Shortest 8-directional path of 0-cells from top-left to bottom-right (path length in cells), or −1.  
**Example.** `[[0,1],[1,0]]` → `2`  
**Spot it.** Unweighted shortest path → BFS with 8 neighbours.

**Ask first**
- 8 directions? → Yes, diagonals count.
- Path length counts cells, not moves? → Cells — so start at 1.

> 💡 **Idea.** Plain BFS from the top-left with 8 directions; the first time you pop the bottom-right, its level is the answer.

**Brute force:** DFS trying all paths, exponential.

**Steps**
1. If the start or end is blocked, return −1.
2. Queue `(0, 0, 1)` and mark it.
3. Pop; if it is the target, return its length.
4. Push unvisited open neighbours in all 8 directions with length + 1.

**Cost:** Time O(n²) · Space O(n²)

**Likely follow-ups**
- **Return the path itself?** Store each cell's parent when you first visit it; walk back from the end.
- **Faster on big open grids?** A* with the Chebyshev distance (`max(|dr|, |dc|)`) as the heuristic — rarely required; mention it.
- **Bidirectional BFS?** Expand from start and end, alternating the smaller frontier, stop when they meet.

In [ ]:
def shortest_path_binary_matrix(grid):
    """Length of the shortest clear path, or -1."""

    n = len(grid)
    if grid[0][0] or grid[-1][-1]:
        return -1
    
    # (row, col, path length in cells)
    queue = deque([(0, 0, 1)])
    grid[0][0] = 1

    while queue:
        r, c, d = queue.popleft()
        # BFS order: the first arrival is the shortest
        if (r, c) == (n - 1, n - 1):
            return d
        
        # all 8 neighbours
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 0:
                    grid[nr][nc] = 1
                    queue.append((nr, nc, d + 1))
    
    return -1


assert shortest_path_binary_matrix([[0, 1], [1, 0]]) == 2
assert shortest_path_binary_matrix([[0, 0, 0], [1, 1, 0], [1, 1, 0]]) == 4
assert shortest_path_binary_matrix([[1, 0, 0], [1, 1, 0], [1, 1, 0]]) == -1
print("ok")

ok


### 1.2.5 [1926. Nearest Exit from Entrance in Maze](https://leetcode.com/problems/nearest-exit-from-entrance-in-maze/) — ★ · Medium

**Problem.** Fewest steps from the entrance to any border empty cell (not the entrance itself).  
**Example.** `maze = [["+","+",".","+"],[".",".",".","+"],["+","+","+","."]], entrance = [1, 2]` → `1`  
**Spot it.** Unweighted shortest path to any of many targets → BFS, stop at the first target.

**Ask first**
- Does the entrance count as an exit if it's on the border? → No.

> 💡 **Idea.** BFS from the entrance; the first border cell (that isn't the entrance) you reach is the nearest exit.

**Brute force:** DFS through every path, exponential.

**Steps**
1. Queue the entrance with distance 0; mark it.
2. Pop; if it is on the border and not the entrance, return its distance.
3. Push unvisited open neighbours with distance + 1.

**Cost:** Time O(rows · cols) · Space O(rows · cols)

**Likely follow-ups**
- **Return the path?** Parent map during BFS, then walk back.
- **Several entrances?** Multi-source BFS from all of them.
- **Some cells are slow (weights)?** Dijkstra.

In [ ]:
def nearest_exit(maze, entrance):
    """Steps to the nearest exit, or -1."""

    rows, cols = len(maze), len(maze[0])
    start = tuple(entrance)

    queue = deque([(start, 0)])
    maze[start[0]][start[1]] = "+"

    while queue:
        (r, c), d = queue.popleft()
        
        # a border cell other than the entrance is an exit
        if (r, c) != start and (r in (0, rows - 1) or c in (0, cols - 1)):
            return d
        
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] == ".":
                # mark as wall once queued
                maze[nr][nc] = "+"
                queue.append(((nr, nc), d + 1))
    return -1


assert nearest_exit([list("++.+"), list("...+"), list("+++.")], [1, 2]) == 1
assert nearest_exit([list("+++"), list("..."), list("+++")], [1, 0]) == 2
assert nearest_exit([list(".+")], [0, 0]) == -1
print("ok")

### 1.2.6 [934. Shortest Bridge](https://leetcode.com/problems/shortest-bridge/) — ★ · Medium

**Problem.** Exactly two islands; fewest 0-cells to flip to connect them.  
**Example.** `[[0,1],[1,0]]` → `1` (flip one 0 to join the two islands)  
**Spot it.** DFS to collect one island, then multi-source BFS outward from all its cells.

**Ask first**
- Exactly two islands? → Yes.

> 💡 **Idea.** DFS to find one island and drop all its cells into a queue. Then BFS outward over water, layer by layer, until you touch the second island — the number of layers is the bridge length.

**Brute force:** try every pair of cells from the two islands, O((m·n)²).

**Steps**
1. Find any land cell; DFS its island, marking cells 2 and queueing them.
2. BFS outward level by level over water, marking visited water 2.
3. The first time a neighbour is original land (1), return the level count.

**Cost:** Time O(n²) · Space O(n²)

**Likely follow-ups**
- **Return the cells of the bridge?** Keep parents during the BFS and walk back from the first cell of island 2.
- **More than two islands — connect them all as cheaply as possible?** Compute the shortest bridge between every pair of islands, then build an MST over the islands.
- **Stack overflow on a big island?** Use iterative DFS (or BFS) for the marking step.

In [ ]:
def shortest_bridge(grid):
    """Zeros to flip to join the two islands."""
    n = len(grid)

    # DFS the first island (ONLY 1), marking it 2 and queueing all its cells
    start = next((r, c) for r in range(n) for c in range(n) if grid[r][c])
    queue = deque()
    stack = [start]
    grid[start[0]][start[1]] = 2

    while stack:
        r, c = stack.pop()
        queue.append((r, c))
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 1:
                grid[nr][nc] = 2
                stack.append((nr, nc))

    # BFS over water; each level is one more flipped cell
    steps = 0
    while queue:
        for _ in range(len(queue)):
            r, c = queue.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n:
                    # touched the second island
                    if grid[nr][nc] == 1:
                        return steps
                    if grid[nr][nc] == 0:
                        grid[nr][nc] = 2
                        queue.append((nr, nc))
        steps += 1
    return -1


assert shortest_bridge([[0, 1], [1, 0]]) == 1
assert shortest_bridge([[0, 1, 0], [0, 0, 0], [0, 0, 1]]) == 2
g = [
    [1, 1, 1, 1, 1],
    [1, 0, 0, 0, 1],
    [1, 0, 1, 0, 1],
    [1, 0, 0, 0, 1],
    [1, 1, 1, 1, 1],
]
assert shortest_bridge(g) == 1
print("ok")

ok


---
# 2) Graph Traversal

**The gist.** For a general graph, first build an adjacency list (node → neighbours), then run the same DFS/BFS as on grids — with a `visited` set, because graphs can have cycles.

> **Think of it like…** exploring a metro map: note which stations connect to which, then walk from your start, ticking off stations so you don't ride the circle line forever.

`graph = defaultdict(list)`; for an undirected edge, add both directions.

**Spot it:** "connected", "provinces", "components", "can you reach / visit all" · "is it a tree" (connected + `n − 1` edges) · "bipartite / two groups" · "clone a graph" · ratios like `a / b = 2` (weighted graph) · "fewest transformations" between words/locks/boards (BFS over states).

**Classic bugs:** adding only one direction for an undirected edge · forgetting nodes that never appear in any edge (loop over `range(n)`) · marking visited too late in BFS.

## 2.1 Adjacency-list traversal

Recipe: build the adjacency list → loop over all nodes → start a traversal from each unvisited one. Number of starts = number of components. What you carry along (a clone map, a colour, an edge flag, a product) is what changes between questions.

### 2.1.1 [133. Clone Graph](https://leetcode.com/problems/clone-graph/) — ★★★ · Medium

**Problem.** Deep-copy a connected undirected graph given one node.  
**Example.** a graph where `1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [1, 3]` → an identical graph made of brand-new nodes  
**Spot it.** Traverse the graph while mapping original → clone (the map doubles as `visited`).

**Ask first**
- Can there be cycles and self-loops? → Yes — the map doubles as the visited set.

> 💡 **Idea.** Map each old node to its copy. DFS/BFS: create the copy the first time you see a node, then hook up neighbours through the map.

**Brute force:** none really — every node and edge must be copied.

**Steps**
1. If the node is already in the map, return its clone.
2. Create the clone and store it in the map.
3. Clone every neighbour recursively and attach them.
4. Return the clone.

**Cost:** Time O(V + E) · Space O(V)

**Likely follow-ups**
- **Graph is disconnected (you get a list of nodes)?** Loop over all nodes and start a traversal from any node not in the map yet, sharing one map.
- **Nodes can have duplicate values?** Key the map by node identity, not by value.
- **Serialise / deserialise the clone?** Write an adjacency list (`id: [neighbour ids]`), rebuild with the same map idea.

In [ ]:
class Node:
    """Graph node with neighbours."""

    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []


def clone_graph(node):
    """Deep copy of the graph reachable from node."""
    # original -> clone; also serves as the visited set
    clones = {}

    def dfs(n):
        if n in clones:
            return clones[n]
        copy = Node(n.val)

        # register before recursing so cycles find this clone
        clones[n] = copy

        copy.neighbors = [dfs(nb) for nb in n.neighbors]
        return copy

    return dfs(node) if node else None


nodes = {i: Node(i) for i in range(1, 5)}
for a, nbs in {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [1, 3]}.items():
    nodes[a].neighbors = [nodes[b] for b in nbs]
    
c = clone_graph(nodes[1])
assert c is not nodes[1] and [n.val for n in c.neighbors] == [2, 4]
assert [n.val for n in c.neighbors[0].neighbors] == [1, 3]
print("ok")

ok


### 2.1.2 [547. Number of Provinces](https://leetcode.com/problems/number-of-provinces/) — ★ · Medium

**Problem.** Adjacency matrix of cities; count connected groups.  
**Example.** `[[1,1,0],[1,1,0],[0,0,1]]` → `2`  
**Spot it.** Connected components on a matrix-encoded graph.

**Ask first**
- Input is an adjacency *matrix*? → Yes, so neighbours are the `1`s in row i (O(n²) total).

> 💡 **Idea.** Each unvisited city starts a DFS that marks its whole province. Count the starts.

**Brute force:** union–find over every 1 in the matrix — same O(n²·α).

**Steps**
1. `seen` array of size n.
2. For each unseen city, count one province and DFS.
3. The DFS visits every `j` with `is_connected[i][j] == 1` that is unseen.

**Cost:** Time O(n²) (every matrix entry is read) · Space O(n)

**Likely follow-ups**
- **Connections arrive one by one, count after each?** Union–find with a component counter; decrement on each successful union.
- **DFS vs union–find — which?** Static graph: DFS is simplest. Edges keep arriving: union–find.
- **Size of the largest province?** Track sizes in union–find or count nodes per DFS.

In [15]:
def find_circle_num(is_connected):
    """Number of connected components."""
    n = len(is_connected)
    seen = [False] * n

    def dfs(i):
        seen[i] = True
        # row i lists the neighbours of city i
        for j in range(n):
            if is_connected[i][j] and not seen[j]:
                dfs(j)

    count = 0
    for i in range(n):
        # an unseen city starts a new province
        if not seen[i]:
            count += 1
            dfs(i)
    return count


assert find_circle_num([[1, 1, 0], [1, 1, 0], [0, 0, 1]]) == 2
assert find_circle_num([[1, 0, 0], [0, 1, 0], [0, 0, 1]]) == 3
print("ok")

ok


### 2.1.3 [841. Keys and Rooms](https://leetcode.com/problems/keys-and-rooms/) — ★ · Medium

**Problem.** Room 0 is open; each room has keys to other rooms. Can you visit all rooms?  
**Example.** `[[1], [2], [3], []]` → `True`; `[[1,3],[3,0,1],[2],[0]]` → `False`  
**Spot it.** Reachability from node 0.

**Ask first**
- Are keys reusable? → Yes, a key opens its room forever.

> 💡 **Idea.** Start in room 0, collect keys, visit every room a key opens (DFS/BFS). All rooms visited → true.

**Brute force:** none — it's already O(rooms + keys).

**Steps**
1. Start with room 0 visited.
2. DFS: for each key in the current room, visit that room if new.
3. Return whether the visited count equals the number of rooms.

**Cost:** Time O(n + E) (every key is read once) · Space O(n)

**Likely follow-ups**
- **Return the order you visited rooms in?** Record each room as it's popped.
- **Which rooms can't be reached?** Those not in `seen` after the traversal.
- **Minimum number of extra keys to open everything?** Count components of the room graph that aren't reachable from room 0 (think strongly connected components / sources in the condensed graph).

In [16]:
def can_visit_all_rooms(rooms):
    """True if every room is reachable from room 0."""
    seen, stack = {0}, [0]
    while stack:
        # each key opens a new room (edge)
        for key in rooms[stack.pop()]:
            if key not in seen:
                seen.add(key)
                stack.append(key)
    return len(seen) == len(rooms)


assert can_visit_all_rooms([[1], [2], [3], []])
assert not can_visit_all_rooms([[1, 3], [3, 0, 1], [2], [0]])
print("ok")

ok


### 2.1.4 [323. Number of Connected Components in an Undirected Graph](https://leetcode.com/problems/number-of-connected-components-in-an-undirected-graph/) — ★★★ · Medium · Premium

**Problem.** `n` nodes and an edge list; count connected components.  
**Example.** `n = 5, edges = [[0,1],[1,2],[3,4]]` → `2`  
**Spot it.** Components → DFS from each unvisited node (or union–find: `n − successful unions`).

**Ask first**
- Nodes with no edges? → Still a component each — loop over `range(n)`, not over edges.

> 💡 **Idea.** Build an adjacency list, start a DFS from every unvisited node, count the starts. (Or union–find: start with n, subtract 1 per successful union.)

**Brute force:** none — O(V + E) is optimal.

**Steps**
1. Build an undirected adjacency list.
2. For each node `0..n−1` not yet seen, count a component and traverse it.
3. Return the count.

**Cost:** Time O(V + E) · Space O(V + E)

**Likely follow-ups**
- **Edges arrive online, answer after each?** Union–find with a counter.
- **Edges can also be removed?** Hard online; offline you can reverse time (process removals as additions backwards) with union–find.
- **Size of each component?** Size array in union–find, or count during DFS.

In [17]:
def count_components(n, edges):
    """Number of connected components."""
    # undirected: store both directions
    graph = defaultdict(list)
    for a, b in edges:
        graph[a].append(b)
        graph[b].append(a)
    seen = set()
    count = 0
    # range(n) also covers nodes with no edges
    for start in range(n):
        if start in seen:
            continue
        count += 1
        stack = [start]
        seen.add(start)
        while stack:
            for nb in graph[stack.pop()]:
                if nb not in seen:
                    seen.add(nb)
                    stack.append(nb)
    return count


assert count_components(5, [[0, 1], [1, 2], [3, 4]]) == 2
assert count_components(5, [[0, 1], [1, 2], [2, 3], [3, 4]]) == 1
print("ok")

ok


### 2.1.5 [261. Graph Valid Tree](https://leetcode.com/problems/graph-valid-tree/) — ★★★ · Medium · Premium

**Problem.** Do `n` nodes and these undirected edges form a tree?  
**Example.** `n = 5, edges = [[0,1],[0,2],[0,3],[1,4]]` → `True`; `[[0,1],[1,2],[2,3],[1,3],[1,4]]` → `False`  
**Spot it.** A tree has exactly `n − 1` edges and is connected (equivalently: no cycle).

**Ask first**
- Undirected edges? → Yes. Directed would need "one root, every other node has one parent" (see follow-up).

> 💡 **Idea.** A tree = exactly `n − 1` edges *and* everything connected. Check the edge count, then check there's one component (DFS or union–find).

**Brute force:** DFS for a cycle plus a connectivity check — also O(V + E).

**Steps**
1. If `len(edges) != n − 1`, return `False`.
2. Otherwise return whether there is exactly one component.

**Cost:** Time O(V + E) · Space O(V + E)

**Likely follow-ups**
- **Why is `n − 1` edges + no cycle enough?** No cycle means a forest; a forest with `n − 1` edges on n nodes has exactly 1 component.
- **Directed version — is it a rooted tree?** Exactly one node with in-degree 0, all others in-degree 1, and everything reachable from the root.
- **Which edge to remove to make it a tree (LC 684)?** The first edge whose union fails.

In [18]:
def valid_tree(n, edges):
    """True if the graph is a tree."""
    # n - 1 edges + connected  =>  no cycle
    return len(edges) == n - 1 and count_components(n, edges) == 1


assert valid_tree(5, [[0, 1], [0, 2], [0, 3], [1, 4]])
assert not valid_tree(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]])
print("ok")

ok


### 2.1.6 [785. Is Graph Bipartite?](https://leetcode.com/problems/is-graph-bipartite/) — ★ · Medium

**Problem.** Can nodes be 2-coloured so every edge joins different colours?  
**Example.** `[[1,3],[0,2],[1,3],[0,2]]` → `True`; `[[1,2,3],[0,2],[0,1,3],[0,2]]` → `False`  
**Spot it.** 2-colouring via BFS/DFS; conflict ⇔ odd cycle.

**Ask first**
- Graph might be disconnected? → Yes — start a BFS from every uncoloured node.

> 💡 **Idea.** Try to 2-colour the graph: give a node a colour, give all its neighbours the other colour. A neighbour with the same colour means not bipartite. Do this for every component.

**Brute force:** try every split into two groups, O(2ⁿ).

**Steps**
1. For each uncoloured node, colour it 0 and traverse its component.
2. Give every uncoloured neighbour the opposite colour.
3. A neighbour that already has the *same* colour → `False`.
4. No conflicts → `True`.

**Cost:** Time O(V + E) · Space O(V)

**Likely follow-ups**
- **Return the two groups?** The colour array is the split: colour 0 vs colour 1.
- **People who dislike each other — can you split them (LC 886)?** Same check on the dislike graph.
- **Three or more colours?** k-colouring is NP-hard for k ≥ 3 — backtracking for small graphs.
- **Union–find version?** For each node, union all its neighbours together; a node in the same set as a neighbour → not bipartite.

In [19]:
def is_bipartite(graph):
    """True if the graph is 2-colourable."""
    color = {}
    # every component needs its own start
    for start in range(len(graph)):
        if start in color:
            continue
        color[start] = 0
        stack = [start]
        while stack:
            node = stack.pop()
            for nb in graph[node]:
                # forced to the opposite colour
                if nb not in color:
                    color[nb] = 1 - color[node]
                    stack.append(nb)
                # same colour across an edge: odd cycle
                elif color[nb] == color[node]:
                    return False
    return True


assert not is_bipartite([[1, 2, 3], [0, 2], [0, 1, 3], [0, 2]])
assert is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]])
print("ok")

ok


### 2.1.7 [1466. Reorder Routes to Make All Paths Lead to the City Zero](https://leetcode.com/problems/reorder-routes-to-make-all-paths-lead-to-the-city-zero/) — ★ · Medium

**Problem.** Tree of directed roads; minimum number of roads to reverse so every city can reach 0.  
**Example.** `n = 6, connections = [[0,1],[1,3],[2,3],[4,0],[4,5]]` → `3`  
**Spot it.** Traverse the undirected tree from 0, counting edges that point away from 0.

**Ask first**
- Is it a tree (n − 1 roads, all connected)? → Yes, so every edge has exactly one correct direction.

> 💡 **Idea.** Treat roads as undirected but remember each one's real direction. DFS from city 0; every edge you cross *in its original direction* (pointing away from 0) must be flipped.

**Brute force:** try flipping subsets of roads, exponential.

**Steps**
1. For each road `a → b`, store `(b, 1)` at `a` and `(a, 0)` at `b`.
2. DFS from 0.
3. Each time you cross a neighbour with flag 1, count a flip.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Why add a reverse edge with cost 0?** So you can walk the whole tree from 0; only original-direction edges cost 1.
- **Not a tree (cycles allowed)?** Becomes a shortest-path problem with 0/1 costs → 0-1 BFS.
- **Every node must reach *every* node instead?** Strongly-connected orientation questions — much harder; mention SCCs.

In [20]:
def min_reorder(n, connections):
    """Number of edges to flip so all reach city 0."""
    # flag 1: the road really goes this way (away from the caller)
    graph = defaultdict(list)
    for a, b in connections:
        graph[a].append((b, 1))
        graph[b].append((a, 0))
    seen, stack, flips = {0}, [0], 0
    while stack:
        node = stack.pop()
        for nb, is_original in graph[node]:
            if nb not in seen:
                seen.add(nb)
                # a road pointing away from city 0 must be reversed
                flips += is_original
                stack.append(nb)
    return flips


assert min_reorder(6, [[0, 1], [1, 3], [2, 3], [4, 0], [4, 5]]) == 3
assert min_reorder(5, [[1, 0], [1, 2], [3, 2], [3, 4]]) == 2
print("ok")

ok


### 2.1.8 [399. Evaluate Division](https://leetcode.com/problems/evaluate-division/) — ★★ · Medium

**Problem.** Given `a / b = k` equations, answer queries `x / y` (or −1).  
**Example.** `equations = [["a","b"],["b","c"]], values = [2.0, 3.0], queries = [["a","c"],["b","a"],["a","e"]]` → `[6.0, 0.5, -1.0]`  
**Spot it.** Weighted graph: edge `a → b` with weight `k`, `b → a` with `1/k`; a query multiplies weights along a path.

**Ask first**
- Unknown variable or no path? → `-1.0`.
- `x / x` for a known x? → 1.0 (unknown x → −1.0).

> 💡 **Idea.** `a / b = 2` is an edge `a → b` weight 2 and `b → a` weight 1/2. A query `x / y` = multiply the weights along any path from x to y.

**Brute force:** DFS per query — fine for small inputs (that's the solution here), O(Q · (V + E)).

**Steps**
1. Add edges `a → b` (k) and `b → a` (1/k).
2. For each query, unknown variables → −1.
3. DFS from `x`, multiplying weights, until `y` is reached.
4. No path → −1.

**Cost:** Time O(Q · (V + E)) · Space O(V + E)

**Likely follow-ups**
- **Lots of queries?** Weighted union–find: store each node's ratio to its root; `x / y = ratio[x] / ratio[y]` if same root. Nearly O(1) per query.
- **Detect contradicting equations (LC 2307)?** Weighted union–find; when two nodes are already connected, check the implied ratio matches (with a tolerance).
- **Precompute all pairs?** Floyd–Warshall with multiplication, O(V³) — fine for few variables.

In [21]:
def calc_equation(equations, values, queries):
    """Answers to division queries."""
    # a / b = k  gives edges a -> b (k) and b -> a (1 / k)
    graph = defaultdict(list)
    for (a, b), k in zip(equations, values):
        graph[a].append((b, k))
        graph[b].append((a, 1 / k))

    def dfs(x, y, seen):
        """Product of ratios along a path x -> y, or -1."""
        if x == y:
            return 1.0
        seen.add(x)
        for nb, w in graph[x]:
            if nb not in seen:
                found = dfs(nb, y, seen)
                if found > 0:
                    return w * found
        return -1.0

    # unknown variables have no answer
    return [dfs(x, y, set()) if x in graph and y in graph else -1.0 for x, y in queries]


res = calc_equation(
    [["a", "b"], ["b", "c"]],
    [2.0, 3.0],
    [["a", "c"], ["b", "a"], ["a", "e"], ["a", "a"], ["x", "x"]],
)
assert res == [6.0, 0.5, -1.0, 1.0, -1.0]
print(res)

[6.0, 0.5, -1.0, 1.0, -1.0]


## 2.2 BFS over state spaces

When nodes are *states* (words, lock codes, board squares), generate neighbours on the fly instead of building the graph. Ask: what's a state, what's one move, and how do I generate moves cheaply?

### 2.2.1 [127. Word Ladder](https://leetcode.com/problems/word-ladder/) — ★★ · Hard

**Problem.** Fewest words in a transformation sequence from `begin` to `end`, changing one letter at a time, each intermediate word in the list.  
**Example.** `begin = "hit", end = "cog", words = ["hot","dot","dog","lot","log","cog"]` → `5` (hit → hot → dot → dog → cog)  
**Spot it.** Shortest path in an implicit graph → BFS; neighbours via wildcard patterns.

**Ask first**
- Is the end word in the list? → If not, return 0 immediately.

> 💡 **Idea.** Each word is a node; words one letter apart are neighbours. BFS from the start word; generate neighbours by trying all 26 letters at each position.

**Brute force:** compare every pair of words to build the graph, O(N²·L).

**Steps**
1. Map each pattern (one letter replaced by `*`) to its words.
2. BFS from `begin` with step count 1.
3. Neighbours = words sharing any of the current word's patterns; skip seen ones.
4. Empty each used bucket (its words are all queued); return the steps at `end`.

**Cost:** Time O(N · L²) (N words × L patterns × O(L) to build each) · Space O(N · L²)

**Likely follow-ups**
- **Faster?** Bidirectional BFS: grow from both ends, always expanding the smaller side.
- **Return all shortest sequences (LC 126)?** BFS to build parent links level by level, then backtrack from the end word.

In [22]:
def ladder_length(begin_word, end_word, word_list):
    """Length of the shortest transformation sequence, or 0."""
    if end_word not in word_list:
        return 0
    # "h*t" -> every word matching it
    buckets = defaultdict(list)
    for w in word_list:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1 :]].append(w)
    queue, seen = deque([(begin_word, 1)]), {begin_word}
    while queue:
        word, steps = queue.popleft()
        if word == end_word:
            return steps
        for i in range(len(word)):
            pattern = word[:i] + "*" + word[i + 1 :]
            for nb in buckets[pattern]:
                if nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
            # every word in this bucket is queued; never scan it again
            buckets[pattern] = []
    return 0


words = ["hot", "dot", "dog", "lot", "log", "cog"]
assert ladder_length("hit", "cog", words) == 5
assert ladder_length("hit", "cog", words[:-1]) == 0
print(ladder_length("hit", "cog", words))

5


### 2.2.2 [433. Minimum Genetic Mutation](https://leetcode.com/problems/minimum-genetic-mutation/) — ★★ · Medium

**Problem.** Fewest single-character mutations (letters A/C/G/T) from `start` to `end`, each step in `bank`.  
**Example.** `start = "AACCGGTT", end = "AACCGGTA", bank = ["AACCGGTA"]` → `1`  
**Spot it.** Word Ladder with a 4-letter alphabet.

**Ask first**
- Is `end` in the bank? → If not, return −1.

> 💡 **Idea.** Word Ladder with a 4-letter alphabet: BFS from `start`, neighbours = one-letter changes that are in the bank.

**Brute force:** DFS trying all mutation orders, exponential.

**Steps**
1. Put the bank in a set.
2. BFS from `start`.
3. For each position and letter, form the mutation; queue it if it is in the bank and unseen.
4. Return the step count at `end`, or −1.

**Cost:** Time O(|bank| · 8 · 4) · Space O(|bank|)

**Likely follow-ups**
- **Faster for a big bank?** Bidirectional BFS from start and end.
- **Return the mutation path?** Parent map during BFS, then walk back.
- **Mutations have different costs?** Dijkstra instead of BFS.

In [23]:
def min_mutation(start, end, bank):
    """Fewest mutations, or -1."""
    valid = set(bank)
    queue, seen = deque([(start, 0)]), {start}
    while queue:
        gene, steps = queue.popleft()
        if gene == end:
            return steps
        # every single-letter change is a candidate neighbour
        for i in range(len(gene)):
            for ch in "ACGT":
                nb = gene[:i] + ch + gene[i + 1 :]
                if nb in valid and nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
    return -1


assert min_mutation("AACCGGTT", "AACCGGTA", ["AACCGGTA"]) == 1
assert min_mutation("AACCGGTT", "AAACGGTA", ["AACCGGTA", "AACCGCTA", "AAACGGTA"]) == 2
print("ok")

ok


### 2.2.3 [752. Open the Lock](https://leetcode.com/problems/open-the-lock/) — ★★ · Medium

**Problem.** 4-wheel lock from `"0000"`; each move turns one wheel by ±1; avoid `deadends`. Fewest moves to `target`?  
**Example.** `deadends = ["0201","0101","0102","1212","2002"], target = "0202"` → `6`  
**Spot it.** BFS over 10⁴ states.

**Ask first**
- Is `0000` itself a deadend? → Then return −1 right away.

> 💡 **Idea.** Each 4-digit code is a node; each move turns one wheel ±1 (8 neighbours). BFS from `0000`, skipping deadends.

**Brute force:** DFS through codes, exponential and doesn't give the minimum.

**Steps**
1. `seen` starts with the deadends; if `"0000"` is one, return −1.
2. BFS from `"0000"`.
3. Each state has 8 neighbours (each wheel ±1, wrapping with `% 10`).
4. Return the steps when `target` is popped.

**Cost:** Time O(10⁴ · 8) · Space O(10⁴)

**Likely follow-ups**
- **Speed it up?** Bidirectional BFS from `0000` and the target — explores about √ of the states.
- **A* heuristic?** Sum over wheels of the circular distance `min(d, 10 − d)` — admissible because each move changes one wheel by 1.
- **Return the sequence of codes?** Parent map.

In [24]:
def open_lock(deadends, target):
    """Fewest turns to reach target, or -1."""
    # deadends behave like already-visited states
    seen = set(deadends)
    if "0000" in seen:
        return -1
    seen.add("0000")
    queue = deque([("0000", 0)])
    while queue:
        state, steps = queue.popleft()
        if state == target:
            return steps
        # turn each wheel one step up or down
        for i in range(4):
            d = int(state[i])
            for nd in ((d + 1) % 10, (d - 1) % 10):
                nb = state[:i] + str(nd) + state[i + 1 :]
                if nb not in seen:
                    seen.add(nb)
                    queue.append((nb, steps + 1))
    return -1


assert open_lock(["0201", "0101", "0102", "1212", "2002"], "0202") == 6
assert open_lock(["8888"], "0009") == 1
assert (
    open_lock(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888")
    == -1
)
print("ok")

ok


### 2.2.4 [909. Snakes and Ladders](https://leetcode.com/problems/snakes-and-ladders/) — ★★ · Medium

**Problem.** Boustrophedon-numbered board; roll 1–6 each move, snakes/ladders teleport. Fewest moves to the last square?  
**Example.** `board = [[-1, -1], [-1, 3]]` → `1`  
**Spot it.** BFS over squares; the tricky part is converting square number → (row, col).

**Ask first**
- Can you chain a ladder into another ladder in one move? → No, only one jump per roll.
- Board numbering? → Boustrophedon from the bottom-left — write a helper to map square → (r, c).

> 💡 **Idea.** Each square is a node; one die roll = one edge to the next 1–6 squares (follow a snake/ladder if you land on one). BFS from square 1.

**Brute force:** DFS through rolls, exponential.

**Steps**
1. `cell(s)` converts a square number to the board value (row from the bottom, alternating direction).
2. BFS from square 1 with 0 moves.
3. For each roll 1–6, follow a snake/ladder if present.
4. Return the moves when square `n²` is popped.

**Cost:** Time O(n²) · Space O(n²)

**Likely follow-ups**
- **Return the actual rolls?** Store the parent square and the roll used; walk back from n².
- **Expected number of rolls with a fair die?** That's a Markov chain / DP on probabilities, not BFS.
- **Weighted moves?** Dijkstra.

In [25]:
def snakes_and_ladders(board):
    """Fewest dice rolls to reach the final square."""
    n = len(board)

    def cell(s):
        """Board value at square s (1-based, boustrophedon from the bottom)."""
        q, r = divmod(s - 1, n)
        row = n - 1 - q
        # even rows from the bottom run left to right, odd rows right to left
        col = r if q % 2 == 0 else n - 1 - r
        return board[row][col]

    queue, seen = deque([(1, 0)]), {1}
    while queue:
        s, moves = queue.popleft()
        if s == n * n:
            return moves
        for nxt in range(s + 1, min(s + 6, n * n) + 1):
            # a snake or ladder moves you immediately
            dest = cell(nxt) if cell(nxt) != -1 else nxt
            if dest not in seen:
                seen.add(dest)
                queue.append((dest, moves + 1))
    return -1


b = [
    [-1, -1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, 35, -1, -1, 13, -1],
    [-1, -1, -1, -1, -1, -1],
    [-1, 15, -1, -1, -1, -1],
]
assert snakes_and_ladders(b) == 4
assert snakes_and_ladders([[-1, -1], [-1, 3]]) == 1
print("ok")

ok


---
# 3) Topological Sort

**The gist.** Line tasks up so each one comes after everything it depends on. Keep taking a task with no remaining prerequisites, do it, and cross it off the lists of tasks waiting on it.

> **Think of it like…** getting dressed: socks before shoes, shirt before jacket. You can always start with something that needs nothing. If two items each need the other first, you'll never finish — that's a cycle.

**Kahn's algorithm** keeps removing nodes with in-degree 0. The removal order is a valid order; anything never removed is on (or behind) a cycle.

```python
indeg = [0] * n; graph = defaultdict(list)
for a, b in edges: graph[a].append(b); indeg[b] += 1
queue = deque(i for i in range(n) if indeg[i] == 0)
order = []
while queue:
    u = queue.popleft(); order.append(u)
    for v in graph[u]:
        indeg[v] -= 1
        if indeg[v] == 0: queue.append(v)
# len(order) < n  ⇒ cycle
```

**Spot it:** "prerequisites", "must come before", "dependencies", "build order", "can you finish all" (= no directed cycle), "alien alphabet order".

**Classic bugs:** flipping the edge direction (`[a, b]` = "b before a" means edge `b → a`) · forgetting nodes with no edges · adding the same edge twice (inflates in-degree).

## 3.1 Kahn's algorithm and DFS ordering

Every question reuses one `topo_order` helper. The real work is building the right edges: which direction, from what data, on which graph (original, reversed, undirected).

### 3.1.1 [207. Course Schedule](https://leetcode.com/problems/course-schedule/) — ★★★ · Medium

**Problem.** `prerequisites[i] = [a, b]` means take `b` before `a`. Can all courses be finished?  
**Example.** `num_courses = 2, prerequisites = [[1, 0]]` → `True`; `[[1, 0], [0, 1]]` → `False`  
**Spot it.** Prerequisites → directed graph; feasible ⇔ no cycle ⇔ topological order covers all.

**Ask first**
- Edge direction: `[a, b]` means take b first? → Yes → edge `b → a`.

> 💡 **Idea.** Can you finish = the prerequisite graph has no cycle. Run Kahn's: if every course gets taken, there's no cycle.

**Brute force:** DFS from every course looking for a cycle without memo, O(V·(V + E)).

**Steps**
1. Edge `b → a` for each prerequisite `[a, b]`; count in-degrees.
2. Queue every course with in-degree 0.
3. Pop a course, append it, decrement its successors; queue those reaching 0.
4. Feasible iff the order contains all courses.

**Cost:** Time O(V + E) · Space O(V + E)

**Likely follow-ups**
- **Minimum number of semesters if you can take any number per semester (LC 1136/2050)?** Kahn's by levels — each BFS level is one semester. With durations: longest path in the DAG.
- **At most k courses per semester (LC 1494)?** Becomes bitmask DP — much harder (small n only).
- **Print the cycle?** DFS with colours (white/grey/black) and a parent pointer; a grey→grey edge closes the cycle.
- **DFS version?** 3-colour DFS; a back edge to a grey node = cycle.

In [26]:
def topo_order(n, edges):
    """Kahn's order for edges u -> v (shorter than n if cyclic)."""
    graph = defaultdict(list)
    indeg = [0] * n
    for u, v in edges:
        graph[u].append(v)
        indeg[v] += 1
    # nodes with nothing left to wait for
    queue = deque(i for i in range(n) if indeg[i] == 0)
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        # u is done: its dependants lose one requirement
        for v in graph[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return order


def can_finish(num_courses, prerequisites):
    """True if the prerequisite graph is acyclic."""
    # [a, b] means b must come before a
    edges = [(b, a) for a, b in prerequisites]
    return len(topo_order(num_courses, edges)) == num_courses


assert can_finish(2, [[1, 0]])
assert not can_finish(2, [[1, 0], [0, 1]])
print("ok")

ok


### 3.1.2 [210. Course Schedule II](https://leetcode.com/problems/course-schedule-ii/) — ★★ · Medium

**Problem.** Return any valid order to take all courses, or `[]` if impossible.  
**Example.** `num_courses = 4, prerequisites = [[1,0],[2,0],[3,1],[3,2]]` → `[0, 1, 2, 3]`  
**Spot it.** Same as Course Schedule, return the order itself.

**Ask first**
- Any valid order OK? → Yes. If they want the lexicographically smallest, use a min-heap instead of a queue.

> 💡 **Idea.** Kahn's algorithm — the order you take courses out of the queue is the answer. Fewer than n taken → cycle → return `[]`.

**Brute force:** try all orders, O(n!).

**Steps**
1. Build edges `b → a`.
2. Run Kahn's algorithm.
3. Return the order if it has all courses, else `[]`.

**Cost:** Time O(V + E) · Space O(V + E)

**Likely follow-ups**
- **Lexicographically smallest order?** Kahn's with a min-heap instead of a FIFO queue: O(V log V + E).
- **All valid orders?** Backtracking over the current zero in-degree nodes — exponential output.
- **Is the order unique?** Unique iff the queue never holds more than one node at a time.

In [27]:
def find_order(num_courses, prerequisites):
    """A valid course order, or empty."""
    order = topo_order(num_courses, [(b, a) for a, b in prerequisites])
    # a short order means some courses sit on a cycle
    return order if len(order) == num_courses else []


assert find_order(2, [[1, 0]]) == [0, 1]
order = find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]])
assert order[0] == 0 and order[-1] == 3
assert find_order(2, [[0, 1], [1, 0]]) == []
print(order)

[0, 1, 2, 3]


### 3.1.3 [269. Alien Dictionary](https://leetcode.com/problems/alien-dictionary/) — ★★★ · Hard · Premium

**Problem.** Words sorted in an unknown alphabet; return a valid letter order (or `""` if invalid).  
**Example.** `["wrt", "wrf", "er", "ett", "rftt"]` → `"wertf"`  
**Spot it.** Adjacent words give ordering constraints → topological sort on letters.

**Ask first**
- A word comes before its own prefix (`"abc"` before `"ab"`)? → Invalid, return `""`.

> 💡 **Idea.** Compare each pair of neighbouring words; the first different letter gives an edge `a → b`. Then topological sort.

**Brute force:** try all letter orders, O(26!).

**Steps**
1. Collect all letters (each is a node, even without edges).
2. For each adjacent pair, the first differing letters give an edge `a → b`.
3. No difference and the first word is longer → invalid, return `""`.
4. Topologically sort; a cycle (short order) → `""`.

**Cost:** Time O(total characters) · Space O(alphabet + edges)

**Likely follow-ups**
- **Is the order unique?** Check the queue never has 2+ letters at once.
- **Verify a given order instead (LC 953)?** Map letter → rank and compare neighbouring words.

In [28]:
def alien_order(words):
    """Letter order consistent with the sorted words."""
    letters = sorted({ch for w in words for ch in w})
    index = {ch: i for i, ch in enumerate(letters)}
    edges = set()
    for w1, w2 in zip(words, words[1:]):
        # only the first differing position carries information
        for a, b in zip(w1, w2):
            if a != b:
                edges.add((index[a], index[b]))
                break
        # no difference: a longer word may not come before its prefix
        else:
            if len(w1) > len(w2):
                return ""
    order = topo_order(len(letters), sorted(edges))
    return "".join(letters[i] for i in order) if len(order) == len(letters) else ""


assert alien_order(["wrt", "wrf", "er", "ett", "rftt"]) == "wertf"
assert alien_order(["z", "x"]) == "zx"
assert alien_order(["z", "x", "z"]) == ""
assert alien_order(["abc", "ab"]) == ""
print(alien_order(["wrt", "wrf", "er", "ett", "rftt"]))

wertf


### 3.1.4 [310. Minimum Height Trees](https://leetcode.com/problems/minimum-height-trees/) — ★★ · Medium

**Problem.** Roots of all minimum-height trees of an undirected tree.  
**Example.** `n = 4, edges = [[1,0],[1,2],[1,3]]` → `[1]`  
**Spot it.** Peel leaves layer by layer (topological sort on an undirected tree); the last 1–2 nodes are the centres.

**Ask first**
- Is it always a tree? → Yes (n − 1 edges, connected).

> 💡 **Idea.** The best roots are the middle of the longest path. Peel leaves layer by layer (like Kahn's on an undirected tree) until 1 or 2 nodes remain — those are the answer.

**Brute force:** BFS from every node to measure its height, O(n²).

**Steps**
1. Build adjacency sets; start with all leaves (degree 1).
2. While more than 2 nodes remain, remove the current leaves.
3. Each removal lowers a neighbour's degree; neighbours reaching degree 1 are the next leaves.
4. The final leaves are the answer.

**Cost:** Time O(n) · Space O(n)

**Likely follow-ups**
- **Why at most 2 roots?** They're the centre of the longest path (diameter): 1 centre if its length is even, 2 if odd.
- **Another way?** BFS from any node to the farthest node A, BFS from A to the farthest B — the middle of the A–B path is the answer.
- **Height from *every* root?** Rerooting DP: one DFS down, one DFS up — O(n).

In [29]:
def find_min_height_trees(n, edges):
    """Centres of the tree."""
    if n == 1:
        return [0]
    graph = defaultdict(set)
    for a, b in edges:
        graph[a].add(b)
        graph[b].add(a)
    leaves = [i for i in range(n) if len(graph[i]) == 1]
    remaining = n
    # peel one full layer of leaves at a time
    while remaining > 2:
        remaining -= len(leaves)
        nxt = []
        for leaf in leaves:
            nb = graph[leaf].pop()
            graph[nb].discard(leaf)
            # the neighbour just became a leaf of the next layer
            if len(graph[nb]) == 1:
                nxt.append(nb)
        leaves = nxt
    return sorted(leaves)


assert find_min_height_trees(4, [[1, 0], [1, 2], [1, 3]]) == [1]
assert find_min_height_trees(6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]]) == [3, 4]
print("ok")

ok


---
# 4) Union-Find

**The gist.** Track which items are in the same group while groups keep merging. Each group has one representative (root); two items are connected iff they share a root.

> **Think of it like…** companies merging: every employee can ask "who's my ultimate parent company?". When two companies merge, one CEO starts reporting to the other — nobody else moves.

With **path compression** (point nodes closer to the root as you search) and **union by size** (hang the smaller tree under the bigger), each operation is basically O(1).

**Spot it:** "redundant edge", "merge accounts / groups", "connected after each new edge", "number of components as edges arrive", undirected cycle detection.

**Classic bugs:** comparing `parent[a] == parent[b]` instead of `find(a) == find(b)` · forgetting to decrement the component count only on a *successful* union.

In [30]:
class DSU:
    """Disjoint set union with path compression and union by size."""

    def __init__(self, n):
        self.parent = list(range(n))
        self.size = [1] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        """Merge groups; False if already joined."""
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        self.size[ra] += self.size[rb]
        return True


d = DSU(4)
print(d.union(0, 1), d.union(1, 0), d.find(1) == d.find(0))

True False True


## 4.1 Dynamic connectivity

The DSU above is the whole toolkit: `union` returns `False` when two items were already connected — that one bit answers most questions.

### 4.1.1 [684. Redundant Connection](https://leetcode.com/problems/redundant-connection/) — ★★ · Medium

**Problem.** A tree plus one extra edge; return the edge that can be removed (the last such in input).  
**Example.** `[[1,2],[1,3],[2,3]]` → `[2, 3]`  
**Spot it.** The first edge joining two already-connected nodes closes the cycle → union–find.

**Ask first**
- Several answers possible? → Return the one that appears last — processing in order and returning the first failing union gives exactly that.

> 💡 **Idea.** Add edges one by one with union–find. The first edge whose two ends are *already* connected closes the cycle — that's the redundant one.

**Brute force:** remove each edge and check if the rest is a tree, O(E·(V + E)).

**Steps**
1. Create a DSU over nodes 1..n.
2. Union each edge in order.
3. The first union that returns `False` is the answer.

**Cost:** Time O(n α(n)) · Space O(n)

**Likely follow-ups**
- **Directed graph (LC 685)?** Three cases: a node with two parents, a cycle, or both. Find the two parent edges, try skipping the second; if a cycle still appears, the answer is the first.
- **Return *all* edges on the cycle?** Find the cycle with DFS (parent pointers) once you know the closing edge.
- **Edges arrive as a stream?** Union–find handles it — check each edge as it arrives.

In [31]:
def find_redundant_connection(edges):
    """Edge that creates the cycle."""
    dsu = DSU(len(edges) + 1)
    for a, b in edges:
        # already connected: this edge closes a cycle
        if not dsu.union(a, b):
            return [a, b]
    return []


assert find_redundant_connection([[1, 2], [1, 3], [2, 3]]) == [2, 3]
assert find_redundant_connection([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]) == [1, 4]
print("ok")

ok


### 4.1.2 [721. Accounts Merge](https://leetcode.com/problems/accounts-merge/) — ★★ · Medium

**Problem.** Merge accounts that share any email; output name + sorted emails.  
**Example.** `[["John","a@x.com","b@x.com"],["John","b@x.com","c@x.com"],["Mary","m@x.com"]]` → `[["John","a@x.com","b@x.com","c@x.com"],["Mary","m@x.com"]]`  
**Spot it.** "Share any element → same group" → union–find on account indices.

**Ask first**
- Can two different people share a name? → Yes — only shared *emails* merge accounts, never names.

> 💡 **Idea.** Union all emails inside each account. Emails with the same root belong to one person — group, sort, and put the name in front.

**Brute force:** compare every pair of accounts and merge repeatedly, O(n²·k).

**Steps**
1. For each email, remember its first owner; union later owners with it.
2. Group emails by `find(owner)`.
3. Output each group's name (from the root account) plus sorted emails.

**Cost:** Time O(N α(N) + N log N) for N emails (sorting dominates) · Space O(N)

**Likely follow-ups**
- **DFS instead of union–find?** Build an email graph (link each account's first email to the others) and take components.
- **Scale it up (millions of accounts)?** Union–find over account ids (not emails) keeps arrays small; sorting each group dominates. Distributed: connected components in MapReduce/Spark.
- **Accounts arrive in a stream?** Union–find updates incrementally.

In [32]:
def accounts_merge(accounts):
    """Merged accounts with sorted emails."""
    dsu = DSU(len(accounts))
    # email -> first account that listed it
    owner = {}
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            # shared email: the two accounts are the same person
            if e in owner:
                dsu.union(i, owner[e])
            else:
                owner[e] = i
    # collect emails under each group's root account
    groups = defaultdict(list)
    for e, i in owner.items():
        groups[dsu.find(i)].append(e)
    return [[accounts[root][0]] + sorted(es) for root, es in groups.items()]


accounts = [
    ["John", "johnsmith@mail.com", "john_newyork@mail.com"],
    ["John", "johnsmith@mail.com", "john00@mail.com"],
    ["Mary", "mary@mail.com"],
    ["John", "johnnybravo@mail.com"],
]
out = sorted(accounts_merge(accounts))
assert out == [
    ["John", "john00@mail.com", "john_newyork@mail.com", "johnsmith@mail.com"],
    ["John", "johnnybravo@mail.com"],
    ["Mary", "mary@mail.com"],
]
print(out)

[['John', 'john00@mail.com', 'john_newyork@mail.com', 'johnsmith@mail.com'], ['John', 'johnnybravo@mail.com'], ['Mary', 'mary@mail.com']]


---
# 5) Weighted Shortest Paths and MST

**The gist.** When edges cost different amounts, "fewest edges" (BFS) isn't "cheapest". Dijkstra always expands the cheapest unfinished node next; with non-negative costs, that node's cost can't improve later, so it's final.

> **Think of it like…** water poured at the source flowing through pipes at constant speed — it reaches each junction first along the cheapest path. Dijkstra simulates that flood with a heap.

```python
dist = {src: 0}; heap = [(0, src)]
while heap:
    d, u = heapq.heappop(heap)
    if d > dist[u]: continue           # stale entry
    for v, w in graph[u]:
        if d + w < dist.get(v, inf):
            dist[v] = d + w; heapq.heappush(heap, (d + w, v))
```

Swap how costs combine (`max` instead of `+`) and the same loop solves bottleneck problems. A **minimum spanning tree** connects all nodes at minimum total cost — Prim's is this same loop using edge weight instead of path length.

**Spot it:** weighted edges + "minimum time / cost" · "minimise the largest step" (bottleneck) · "at most k stops" (Bellman–Ford) · "connect all points cheaply" (MST).

**Classic bugs:** not skipping stale heap entries · using Dijkstra with negative weights · in k-stop Bellman–Ford, relaxing from the *current* round's array instead of a copy.

## 5.1 Dijkstra and variants

Dijkstra's loop stays the same; what changes is how a path's cost combines (`+`, `max`, `×`) and when you can stop (first time the target is popped).

### 5.1.1 [743. Network Delay Time](https://leetcode.com/problems/network-delay-time/) — ★★ · Medium

**Problem.** Directed weighted edges; time for a signal from `k` to reach all nodes (−1 if not all).  
**Example.** `times = [[2,1,1],[2,3,1],[3,4,1]], n = 4, k = 2` → `2`  
**Spot it.** Single-source shortest paths, non-negative weights → Dijkstra.

**Ask first**
- Can weights be negative? → No (if they could, use Bellman–Ford).

> 💡 **Idea.** Dijkstra from `k`. The answer is the biggest final distance; if some node is never reached, −1.

**Brute force:** Bellman–Ford, O(V·E).

**Steps**
1. Build the adjacency list.
2. Dijkstra from `k`: pop the closest node, skip stale entries, relax its edges.
3. If not every node got a distance, return −1.
4. Otherwise return the maximum distance.

**Cost:** Time O(E log V) · Space O(V + E)

**Likely follow-ups**
- **Negative weights?** Bellman–Ford: relax all edges n − 1 times; a further improvement means a negative cycle.
- **Print the path to a node?** Store `parent[v] = u` on each relaxation, walk back.
- **Dense graph?** Array-based Dijkstra without a heap: O(V²).
- **All pairs?** Floyd–Warshall O(V³) for small V; or Dijkstra from every node.

In [33]:
def network_delay_time(times, n, k):
    """Time for all nodes to receive the signal."""
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = {k: 0}
    heap = [(0, k)]
    while heap:
        # closest unfinished node: its distance is final
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            continue
        # relax outgoing edges
        for v, w in graph[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
    # the last node to hear the signal decides the time
    return max(dist.values()) if len(dist) == n else -1


assert network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2) == 2
assert network_delay_time([[1, 2, 1]], 2, 2) == -1
print("ok")

ok


### 5.1.2 [1631. Path With Minimum Effort](https://leetcode.com/problems/path-with-minimum-effort/) — ★★ · Medium

**Problem.** Grid of heights; a path's effort is its largest absolute step difference. Minimise effort from top-left to bottom-right.  
**Example.** `[[1,2,2],[3,8,2],[5,3,5]]` → `2`  
**Spot it.** Bottleneck shortest path → Dijkstra where path cost = max edge, not sum.

**Ask first**
- Effort = max absolute height difference along the path? → Yes (not the sum).

> 💡 **Idea.** A path's cost is its *biggest* step. Dijkstra still works — just combine with `max(d, step)` instead of `d + step`.

**Brute force:** try all paths, exponential.

**Steps**
1. Heap of `(effort, r, c)` starting at `(0, 0, 0)`.
2. Pop; if it is the target, return the effort.
3. Skip stale entries.
4. For each neighbour, new effort = `max(effort, |height difference|)`; push if it improves.

**Cost:** Time O(mn log(mn)) · Space O(mn)

**Likely follow-ups**
- **Binary search version?** Binary-search the effort E; check with BFS/DFS using only steps ≤ E. O(m·n·log(max height)).
- **Union–find version?** Sort all edges by weight, union them in order until start and end connect; the last weight added is the answer.
- **Minimise the *sum* of steps instead?** Normal Dijkstra with `+`.

In [34]:
def minimum_effort_path(heights):
    """Least possible maximum step on a path."""
    rows, cols = len(heights), len(heights[0])
    best = {(0, 0): 0}
    heap = [(0, 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        # first time the target is popped, its effort is optimal
        if (r, c) == (rows - 1, cols - 1):
            return d
        if d > best[(r, c)]:
            continue
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                # path cost is the largest single step so far
                nd = max(d, abs(heights[nr][nc] - heights[r][c]))
                if nd < best.get((nr, nc), float("inf")):
                    best[(nr, nc)] = nd
                    heapq.heappush(heap, (nd, nr, nc))
    return 0


assert minimum_effort_path([[1, 2, 2], [3, 8, 2], [5, 3, 5]]) == 2
assert minimum_effort_path([[1, 2, 3], [3, 8, 4], [5, 3, 5]]) == 1
print("ok")

ok


### 5.1.3 [787. Cheapest Flights Within K Stops](https://leetcode.com/problems/cheapest-flights-within-k-stops/) — ★★ · Medium

**Problem.** Cheapest price from `src` to `dst` using at most `k` stops (k + 1 flights).  
**Example.** `n = 4, flights = [[0,1,100],[1,2,100],[2,0,100],[1,3,600],[2,3,200]], src = 0, dst = 3, k = 1` → `700`  
**Spot it.** Edge-count limit → Bellman–Ford for `k + 1` rounds.

**Ask first**
- Stops = cities in between, not edges? → Yes, so k stops = k + 1 flights.

> 💡 **Idea.** At most k stops = at most k + 1 edges. Run Bellman–Ford for k + 1 rounds, each round relaxing from a *copy* of last round's prices.

**Brute force:** DFS over all routes with ≤ k stops, exponential.

**Steps**
1. `prices[src] = 0`, everything else ∞.
2. Repeat `k + 1` times: copy prices, relax every flight using the old prices into the copy.
3. Replace prices with the copy.
4. Return `prices[dst]`, or −1 if still ∞.

**Cost:** Time O(k · E) · Space O(n)

**Likely follow-ups**
- **Dijkstra version?** Heap of `(cost, city, stops)`; allow revisiting a city if you arrive with fewer stops. First pop of `dst` is the answer.
- **Return the route?** Keep a parent per `(city, round)` (or carry the path in the heap state) and walk back.
- **Why the copy each round?** Without it, one round could chain several flights and break the stop limit.

In [35]:
def find_cheapest_price(n, flights, src, dst, k):
    """Cheapest cost with at most k stops, or -1."""
    inf = float("inf")
    prices = [inf] * n
    prices[src] = 0
    # round i allows paths with at most i flights
    for _ in range(k + 1):
        # read last round's prices, write into a copy
        nxt = prices.copy()
        for u, v, w in flights:
            if prices[u] + w < nxt[v]:
                nxt[v] = prices[u] + w
        prices = nxt
    return -1 if prices[dst] == inf else prices[dst]


flights = [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]]
assert find_cheapest_price(4, flights, 0, 3, 1) == 700
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1) == 200
assert find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0) == 500
print("ok")

ok


### 5.1.4 [1584. Min Cost to Connect All Points](https://leetcode.com/problems/min-cost-to-connect-all-points/) — ★★ · Medium

**Problem.** Connect all points with minimum total Manhattan distance.  
**Example.** `[[0,0],[2,2],[3,10],[5,2],[7,0]]` → `20`  
**Spot it.** Minimum spanning tree on a complete graph → Prim's algorithm.

**Ask first**
- Every pair can connect? → Yes — a complete graph, so O(n²) edges.

> 💡 **Idea.** It's a minimum spanning tree. Prim's: grow the tree from any point, always adding the cheapest point not in it yet (Manhattan distances, computed on the fly).

**Brute force:** Kruskal: build all n² edges, sort, union — O(n² log n).

**Steps**
1. `best[i]` = cheapest edge from the tree to point `i`; start with point 0 at cost 0.
2. Pick the outside point with the smallest `best`; add it and its cost.
3. Update `best` for all outside points using distances from the new point.
4. Repeat n times.

**Cost:** Time O(n²) · Space O(n)

**Likely follow-ups**
- **Kruskal vs Prim here?** Dense graph (all pairs) → array-based Prim O(n²) wins; Kruskal is better for a sparse explicit edge list.
- **Return the edges used?** Track which tree point each new point was connected through.
- **Huge number of points?** Only O(n) candidate edges matter for Manhattan MST (sweep by octants) → O(n log n). Rarely asked — mention it.

In [36]:
def min_cost_connect_points(points):
    """MST weight under Manhattan distance."""
    n = len(points)
    # cheapest known edge from the tree to each point
    best = [float("inf")] * n
    in_tree = [False] * n
    best[0] = 0
    total = 0
    for _ in range(n):
        # closest point not yet in the tree
        u = min((i for i in range(n) if not in_tree[i]), key=best.__getitem__)
        in_tree[u] = True
        total += best[u]
        # the new point may offer cheaper edges to the rest
        for v in range(n):
            if not in_tree[v]:
                d = abs(points[u][0] - points[v][0]) + abs(points[u][1] - points[v][1])
                best[v] = min(best[v], d)
    return total


assert min_cost_connect_points([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]) == 20
assert min_cost_connect_points([[3, 12], [-2, 5], [-4, 1]]) == 18
print("ok")

ok


---
# 6) Backtracking

**The gist.** Build answers one choice at a time. After each choice, explore everything that follows, then *undo* it and try the next choice. It's a DFS over a tree of decisions; pruning cuts branches that can't work.

> **Think of it like…** walking a hedge maze with a ball of string: take a path at each fork; at a dead end, wind the string back to the last fork and try the next path.

```python
def backtrack(start, path):
    if is_complete(path):
        out.append(path.copy()); return
    for i in range(start, len(choices)):
        if not valid(choices[i]): continue          # prune
        path.append(choices[i])                     # choose
        backtrack(i + 1, path)                      # explore (i for reuse, 0 for permutations)
        path.pop()                                  # un-choose
```

| Problem type | Recurse with | Avoid duplicates by |
|---|---|---|
| subsets / combinations | `i + 1` | sort + skip `nums[i] == nums[i-1]` when `i > start` |
| unlimited reuse | `i` | — |
| permutations | loop from 0 with a `used` array | sort + skip if the previous equal value is unused |

**Spot it:** "all subsets / combinations / permutations", "generate every valid…", "partition into…", small n (≤ 15–20), board puzzles.

**Classic bugs:** appending `path` instead of `path.copy()` · forgetting to undo a choice · deduping at the wrong level (skip only *siblings*, not the whole tree).

## 6.1 Subsets, combinations and permutations

Three questions pick the template: can an element be reused (`i` vs `i + 1`)? does order matter (start at 0 with `used`, or at `start`)? can the input have duplicates (sort + skip equal siblings)?

### 6.1.1 [78. Subsets](https://leetcode.com/problems/subsets/) — ★★ · Medium

**Problem.** All subsets of distinct integers.  
**Example.** `[1, 2, 3]` → `[[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]`  
**Spot it.** Each element is in or out → backtracking (or iterate: double the list per element).

**Ask first**
- Distinct numbers? → Yes. With duplicates it's LC 90 (sort + skip).

> 💡 **Idea.** For each number: take it or skip it. Every node of the decision tree is a subset — record it.

**Brute force:** it's already exponential — 2ⁿ subsets is the output size.

**Steps**
1. Record a copy of the current path.
2. For each index `i` from `start`, append `nums[i]`.
3. Recurse with `i + 1`.
4. Pop to undo.

**Cost:** Time O(n · 2ⁿ) (2ⁿ subsets, each copied in O(n)) · Space O(n) recursion (plus the output)

**Likely follow-ups**
- **Iterative version?** Start with `[[]]`; for each number, add it to a copy of every existing subset.
- **Bitmask version?** For `mask` in `0..2ⁿ−1`, include `nums[i]` when bit i is set.
- **Only subsets of size k?** That's Combinations (LC 77) — stop when `len(path) == k`.

In [37]:
def subsets(nums):
    """Power set."""
    out = []
    path = []

    def backtrack(start):
        # every node of the decision tree is a subset
        out.append(path.copy())
        for i in range(start, len(nums)):
            # choose, explore, un-choose
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


assert subsets([1, 2, 3]) == [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
print(len(subsets([1, 2, 3, 4])))

16


### 6.1.2 [90. Subsets II](https://leetcode.com/problems/subsets-ii/) — ★★ · Medium

**Problem.** All unique subsets when the input has duplicates.  
**Example.** `[1, 2, 2]` → `[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]`  
**Spot it.** Sort, then skip equal siblings at the same depth.

**Ask first**
- Can the input have duplicates? → Yes, that's the whole point.

> 💡 **Idea.** Sort, then in each loop skip a number equal to the one before it at the *same level* (`i > start and nums[i] == nums[i-1]`).

**Brute force:** generate all subsets and dedupe with a set of tuples — wasteful.

**Steps**
1. Sort.
2. Record the path.
3. Loop `i` from `start`; skip if `i > start` and `nums[i] == nums[i − 1]`.
4. Choose, recurse with `i + 1`, un-choose.

**Cost:** Time O(n · 2ⁿ) · Space O(n)

**Likely follow-ups**
- **Why `i > start` and not `i > 0`?** You may use equal numbers at *different* depths (e.g. `[1, 2, 2]`), just not as two siblings.
- **Iterative version?** When a number repeats, only extend the subsets created in the previous step.

In [38]:
def subsets_with_dup(nums):
    """Unique subsets of a multiset."""
    # equal values become neighbours
    nums = sorted(nums)
    out = []
    path = []

    def backtrack(start):
        out.append(path.copy())
        for i in range(start, len(nums)):
            # same value as the previous sibling: same subtree, skip
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()

    backtrack(0)
    return out


assert subsets_with_dup([1, 2, 2]) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
print(subsets_with_dup([1, 2, 2]))

[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


### 6.1.3 [77. Combinations](https://leetcode.com/problems/combinations/) — ★★ · Medium

**Problem.** All combinations of `k` numbers from `1..n`.  
**Example.** `n = 4, k = 2` → `[[1,2],[1,3],[1,4],[2,3],[2,4],[3,4]]`  
**Spot it.** Subsets restricted to size `k`, with pruning.

**Ask first**
- Order inside a combination matters? → No, `[1,2]` = `[2,1]`.

> 💡 **Idea.** Pick numbers in increasing order (`start` grows), stop when you have k. Prune when there aren't enough numbers left.

**Brute force:** generate all subsets, keep size k — 2ⁿ instead of C(n, k).

**Steps**
1. If the path has `k` numbers, record it.
2. Loop `i` from `start` up to the last value that still leaves room for the remaining slots.
3. Choose `i`, recurse with `i + 1`, un-choose.

**Cost:** Time O(k · C(n, k)) · Space O(k)

**Likely follow-ups**
- **Prune harder?** Loop `i` only up to `n − (k − len(path)) + 1`.
- **Iterative / next combination?** Treat it like next permutation on index arrays, or use bitmasks with popcount k.
- **Just count them?** C(n, k) = n! / (k!·(n−k)!).

In [39]:
def combine(n, k):
    """All k-combinations of 1..n."""
    out = []
    path = []

    def backtrack(start):
        if len(path) == k:
            out.append(path.copy())
            return
        # stop once too few numbers remain to fill the k slots
        for i in range(start, n - (k - len(path)) + 2):
            path.append(i)
            backtrack(i + 1)
            path.pop()

    backtrack(1)
    return out


assert combine(4, 2) == [[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]
assert combine(1, 1) == [[1]]
print(combine(4, 2))

[[1, 2], [1, 3], [1, 4], [2, 3], [2, 4], [3, 4]]


### 6.1.4 [39. Combination Sum](https://leetcode.com/problems/combination-sum/) — ★★ · Medium

**Problem.** Distinct candidates, unlimited reuse; all combinations summing to `target`.  
**Example.** `candidates = [2, 3, 6, 7], target = 7` → `[[2, 2, 3], [7]]`  
**Spot it.** Reuse allowed → recurse with `i` (not `i + 1`); sorted input lets you prune.

**Ask first**
- Can numbers be reused? → Yes, unlimited.
- Are candidates distinct and positive? → Yes (positive makes pruning safe).

> 💡 **Idea.** Backtrack with a running total; recurse with `i` (not `i + 1`) so a number can be reused. Stop when the total passes the target.

**Brute force:** try every multiset up to the target, no pruning.

**Steps**
1. Sort the candidates.
2. Remaining 0 → record the path.
3. For each `i` from `start`: if the candidate exceeds the remaining sum, break.
4. Choose it, recurse with `i` (reuse) and the reduced sum, un-choose.

**Cost:** Time Exponential, bounded by about `n^(target / min)` · Space O(target / min) recursion

**Likely follow-ups**
- **Only count the combinations?** Coin Change II DP: `dp[t] += dp[t − c]`, coins in the outer loop. O(n·target).
- **Each number usable at most once (LC 40)?** Recurse with `i + 1`, sort, skip equal siblings.
- **Order matters (LC 377)?** DP with the target in the outer loop.
- **Limit on how many times each can be used?** Bounded knapsack: loop count 0..limit for each candidate.

In [40]:
def combination_sum(candidates, target):
    """Combinations with reuse summing to target."""
    candidates = sorted(candidates)
    out = []
    path = []

    def backtrack(start, remaining):
        if remaining == 0:
            out.append(path.copy())
            return
        for i in range(start, len(candidates)):
            # sorted: this and every later candidate is too big
            if candidates[i] > remaining:
                break
            path.append(candidates[i])
            # i, not i + 1: the same candidate may be used again
            backtrack(i, remaining - candidates[i])
            path.pop()

    backtrack(0, target)
    return out


assert combination_sum([2, 3, 6, 7], 7) == [[2, 2, 3], [7]]
assert combination_sum([2, 3, 5], 8) == [[2, 2, 2, 2], [2, 3, 3], [3, 5]]
assert combination_sum([2], 1) == []
print(combination_sum([2, 3, 6, 7], 7))

[[2, 2, 3], [7]]


### 6.1.5 [40. Combination Sum II](https://leetcode.com/problems/combination-sum-ii/) — ★★ · Medium

**Problem.** Candidates may repeat; each used at most once; unique combinations summing to `target`.  
**Example.** `candidates = [10, 1, 2, 7, 6, 1, 5], target = 8` → `[[1,1,6],[1,2,5],[1,7],[2,6]]`  
**Spot it.** No reuse (`i + 1`) + duplicate skipping (Subsets II rule).

**Ask first**
- Can the input have duplicates? → Yes, but each *combination* must be unique.

> 💡 **Idea.** Like Combination Sum, but each number once (`i + 1`) and the input has duplicates — sort and skip equal siblings.

**Brute force:** generate everything and dedupe with a set.

**Steps**
1. Sort.
2. Remaining 0 → record.
3. Skip `i > start` with the same value as `i − 1`; break if the candidate exceeds the remainder.
4. Choose, recurse with `i + 1`, un-choose.

**Cost:** Time O(2ⁿ) worst case · Space O(n)

**Likely follow-ups**
- **Count only?** 0/1 knapsack count over distinct values with their multiplicities.
- **Early stop?** Since the list is sorted, break the loop as soon as a number exceeds the remaining target.

In [41]:
def combination_sum2(candidates, target):
    """Unique single-use combinations summing to target."""
    candidates = sorted(candidates)
    out = []
    path = []

    def backtrack(start, remaining):
        if remaining == 0:
            out.append(path.copy())
            return
        for i in range(start, len(candidates)):
            # equal sibling would repeat a combination
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remaining:
                break
            path.append(candidates[i])
            # i + 1: each candidate used at most once
            backtrack(i + 1, remaining - candidates[i])
            path.pop()

    backtrack(0, target)
    return out


assert combination_sum2([10, 1, 2, 7, 6, 1, 5], 8) == [
    [1, 1, 6],
    [1, 2, 5],
    [1, 7],
    [2, 6],
]
assert combination_sum2([2, 5, 2, 1, 2], 5) == [[1, 2, 2], [5]]
print(combination_sum2([10, 1, 2, 7, 6, 1, 5], 8))

[[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]


### 6.1.6 [216. Combination Sum III](https://leetcode.com/problems/combination-sum-iii/) — ★ · Medium

**Problem.** All combinations of `k` distinct digits 1–9 summing to `n`.  
**Example.** `k = 3, n = 7` → `[[1, 2, 4]]`  
**Spot it.** Combinations with both a size and a sum constraint.

**Ask first**
- Digits 1–9, each once? → Yes.

> 💡 **Idea.** Pick k distinct numbers from 1–9 in increasing order with a running sum; prune when the sum overshoots or numbers run out.

**Brute force:** check all 2⁹ subsets — fine here (only 512).

**Steps**
1. Path has `k` digits → record if remaining is 0, then return.
2. Loop digits from `start` to 9; break if a digit exceeds the remainder.
3. Choose, recurse with `d + 1`, un-choose.

**Cost:** Time O(C(9, k)) · Space O(k)

**Likely follow-ups**
- **Bitmask brute force?** Loop all 512 masks, keep those with popcount k and sum n — totally fine.
- **Bigger range (1..m)?** Prune with min/max possible sums of the remaining picks.

In [42]:
def combination_sum3(k, n):
    """k distinct digits summing to n."""
    out = []
    path = []

    def backtrack(start, remaining):
        # size reached: valid only if the sum is exact
        if len(path) == k:
            if remaining == 0:
                out.append(path.copy())
            return
        for d in range(start, 10):
            if d > remaining:
                break
            path.append(d)
            backtrack(d + 1, remaining - d)
            path.pop()

    backtrack(1, n)
    return out


assert combination_sum3(3, 7) == [[1, 2, 4]]
assert combination_sum3(3, 9) == [[1, 2, 6], [1, 3, 5], [2, 3, 4]]
assert combination_sum3(4, 1) == []
print(combination_sum3(3, 9))

[[1, 2, 6], [1, 3, 5], [2, 3, 4]]


### 6.1.7 [46. Permutations](https://leetcode.com/problems/permutations/) — ★★ · Medium

**Problem.** All permutations of distinct integers.  
**Example.** `[1, 2, 3]` → `[[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]]`  
**Spot it.** Order matters → loop from 0 each level, skip used elements.

**Ask first**
- Distinct numbers? → Yes. Duplicates → LC 47.

> 💡 **Idea.** At each position, try every number not used yet (a `used` array), recurse, then un-use it.

**Brute force:** none — n! permutations is the output size.

**Steps**
1. Path length `n` → record.
2. For each index not yet used, mark it and append its value.
3. Recurse.
4. Pop and unmark.

**Cost:** Time O(n · n!) (n! permutations, each copied in O(n)) · Space O(n)

**Likely follow-ups**
- **Iterative?** Insert each new number into every position of every existing permutation, or loop next permutation from the sorted order.
- **Only the k-th permutation (LC 60)?** Factorial number system: pick `digits[(k−1) // (n−1)!]`, repeat.
- **In-place swaps instead of `used`?** Swap `i` into position `start`, recurse, swap back.

In [43]:
def permute(nums):
    """All orderings of nums."""
    out = []
    path = []
    used = [False] * len(nums)

    def backtrack():
        if len(path) == len(nums):
            out.append(path.copy())
            return
        # any unused element may take the next position
        for i, x in enumerate(nums):
            if not used[i]:
                used[i] = True
                path.append(x)
                backtrack()
                path.pop()
                used[i] = False

    backtrack()
    return out


assert permute([1, 2, 3]) == [
    [1, 2, 3],
    [1, 3, 2],
    [2, 1, 3],
    [2, 3, 1],
    [3, 1, 2],
    [3, 2, 1],
]
assert permute([1]) == [[1]]
print(len(permute([1, 2, 3, 4])))

24


### 6.1.8 [47. Permutations II](https://leetcode.com/problems/permutations-ii/) — ★★ · Medium

**Problem.** All unique permutations when the input has duplicates.  
**Example.** `[1, 1, 2]` → `[[1,1,2],[1,2,1],[2,1,1]]`  
**Spot it.** Sort + skip a value equal to the previous one if the previous copy is unused.

**Ask first**
- Can the input have duplicates? → Yes.

> 💡 **Idea.** Sort, then skip `nums[i]` if it equals `nums[i-1]` and `nums[i-1]` isn't used — this makes equal values always appear in left-to-right order, so each permutation is made once.

**Brute force:** generate all n! and dedupe with a set.

**Steps**
1. Sort.
2. Skip index `i` if it is used.
3. Also skip if `nums[i] == nums[i − 1]` and `i − 1` is *not* used (its earlier copy must go first).
4. Choose, recurse, un-choose.

**Cost:** Time O(n · n!) worst case · Space O(n)

**Likely follow-ups**
- **Why `not used[i-1]`?** It forces equal copies to be placed in their original order; otherwise swapping two equal copies makes a duplicate.
- **Counter-based version?** Recurse over distinct values with remaining counts — no sorting trick needed.
- **How many unique permutations?** n! / (c₁!·c₂!·…).

In [44]:
def permute_unique(nums):
    """Unique orderings of a multiset."""
    nums = sorted(nums)
    out = []
    path = []
    used = [False] * len(nums)

    def backtrack():
        if len(path) == len(nums):
            out.append(path.copy())
            return
        for i, x in enumerate(nums):
            # equal values must be taken left to right
            if used[i] or (i and nums[i] == nums[i - 1] and not used[i - 1]):
                continue
            used[i] = True
            path.append(x)
            backtrack()
            path.pop()
            used[i] = False

    backtrack()
    return out


assert permute_unique([1, 1, 2]) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert len(permute_unique([1, 2, 3])) == 6
print(permute_unique([1, 1, 2]))

[[1, 1, 2], [1, 2, 1], [2, 1, 1]]


### 6.1.9 [17. Letter Combinations of a Phone Number](https://leetcode.com/problems/letter-combinations-of-a-phone-number/) — ★★ · Medium

**Problem.** All letter strings a phone digit string could represent.  
**Example.** `"23"` → `["ad","ae","af","bd","be","bf","cd","ce","cf"]`  
**Spot it.** One choice per position from a fixed mapping → backtracking (Cartesian product).

**Ask first**
- Empty input? → Return `[]`, not `[""]`.

> 💡 **Idea.** Each digit offers 3–4 letters; build strings digit by digit, trying each letter.

**Brute force:** it's already the output size, 4ⁿ.

**Steps**
1. Empty input → `[]`.
2. At position `i`, try each letter of `digits[i]`.
3. Recurse with `i + 1` and the extended string.
4. At the end of the digits, record the string.

**Cost:** Time O(4ⁿ · n) · Space O(n) recursion

**Likely follow-ups**
- **Iterative?** Start with `[""]`; for each digit, replace the list with every string + every letter (BFS).
- **Only real words from a dictionary?** Build a trie of the dictionary and prune any prefix that isn't in it.
- **Split into two (or more) dictionary words?** When you reach a word end in the trie, either stop or jump back to the trie root and continue.

In [45]:
PHONE = {
    "2": "abc",
    "3": "def",
    "4": "ghi",
    "5": "jkl",
    "6": "mno",
    "7": "pqrs",
    "8": "tuv",
    "9": "wxyz",
}


def letter_combinations(digits):
    """Letter strings for a digit string."""
    if not digits:
        return []
    out = []

    def backtrack(i, path):
        if i == len(digits):
            out.append(path)
            return
        # each letter of this digit is one branch
        for ch in PHONE[digits[i]]:
            backtrack(i + 1, path + ch)

    backtrack(0, "")
    return out


assert letter_combinations("23") == [
    "ad",
    "ae",
    "af",
    "bd",
    "be",
    "bf",
    "cd",
    "ce",
    "cf",
]
assert letter_combinations("") == []
print(letter_combinations("23"))

['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']


### 6.1.10 [22. Generate Parentheses](https://leetcode.com/problems/generate-parentheses/) — ★★ · Medium

**Problem.** All well-formed strings of `n` pairs of parentheses.  
**Example.** `n = 3` → `["((()))", "(()())", "(())()", "()(())", "()()()"]`  
**Spot it.** Choices constrained by counts → backtrack with `open` and `close` counters.

**Ask first**
- Return all of them? → Yes, in any order.

> 💡 **Idea.** Add `(` while you still have some left; add `)` only while it doesn't outnumber `(`. Every finished string is valid.

**Brute force:** generate all 2²ⁿ strings and validate each.

**Steps**
1. Length `2n` → record.
2. If `open < n`, add `(`.
3. If `close < open`, add `)`.

**Cost:** Time O(4ⁿ / √n) (the Catalan number of results) · Space O(n)

**Likely follow-ups**
- **How many are there?** The n-th Catalan number: C(2n, n) / (n + 1).
- **DP version?** Every string = `(` + A + `)` + B, where A has i pairs and B has n − 1 − i.
- **Several bracket types?** Keep a stack of open types; a closer must match the top.

In [46]:
def generate_parenthesis(n):
    """All balanced strings of n pairs."""
    out = []

    def backtrack(path, open_count, close_count):
        if len(path) == 2 * n:
            out.append(path)
            return
        # an opener is allowed while some remain
        if open_count < n:
            backtrack(path + "(", open_count + 1, close_count)
        # a closer is allowed only if it has an opener to match
        if close_count < open_count:
            backtrack(path + ")", open_count, close_count + 1)

    backtrack("", 0, 0)
    return out


assert generate_parenthesis(3) == ["((()))", "(()())", "(())()", "()(())", "()()()"]
assert generate_parenthesis(1) == ["()"]
print(generate_parenthesis(3))

['((()))', '(()())', '(())()', '()(())', '()()()']


## 6.2 Search with constraints

Choices are limited by a board, a string or a running value. The skill: check validity in O(1) (sets, bitmasks, counters) and prune *before* recursing.

### 6.2.1 [79. Word Search](https://leetcode.com/problems/word-search/) — ★★★ · Medium

**Problem.** Does the word exist in the grid as a path of adjacent cells (no reuse)?  
**Example.** `board = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]], word = "ABCCED"` → `True`  
**Spot it.** Path search with "no revisit" → DFS with temporary marking and backtracking.

**Ask first**
- Can a cell be used twice in one word? → No → mark on enter, unmark on exit.

> 💡 **Idea.** From each cell that matches the first letter, DFS letter by letter. Mark a cell while it's on the current path, unmark it when you back out.

**Brute force:** none much better — O(m·n·3^L) is expected.

**Steps**
1. From each cell, DFS with index `i` into the word.
2. Out of bounds or letter mismatch → false; `i == len(word)` → true.
3. Mark the cell, try the 4 neighbours with `i + 1`.
4. Unmark the cell and return the result.

**Cost:** Time O(cells · 3^L) · Space O(L) recursion

**Likely follow-ups**
- **Search pruning for big boards?** Count letters first — if the board lacks enough of a letter, return False. Start from the rarer end of the word (reverse it if the last letter is rarer).
- **Many words at once (LC 212)?** Put the words in a trie and do one DFS per cell walking the trie.
- **No extra memory for visited?** Temporarily overwrite the cell with `#` and restore it.

In [47]:
def exist(board, word):
    """True if word can be traced in the grid."""
    rows, cols = len(board), len(board[0])

    def dfs(r, c, i):
        if i == len(word):
            return True
        if not (0 <= r < rows and 0 <= c < cols) or board[r][c] != word[i]:
            return False
        # mark as used on the current path
        board[r][c] = "#"
        found = any(dfs(r + dr, c + dc, i + 1) for dr, dc in DIRS)
        # free the cell for other paths
        board[r][c] = word[i]
        return found

    return any(dfs(r, c, 0) for r in range(rows) for c in range(cols))


b = [list("ABCE"), list("SFCS"), list("ADEE")]
assert exist(b, "ABCCED") and exist(b, "SEE") and not exist(b, "ABCB")
print("ok")

ok


### 6.2.2 [131. Palindrome Partitioning](https://leetcode.com/problems/palindrome-partitioning/) — ★★ · Medium

**Problem.** All ways to split `s` into palindromic pieces.  
**Example.** `"aab"` → `[["a", "a", "b"], ["aa", "b"]]`  
**Spot it.** Choose the next cut; only continue if the piece is a palindrome.

**Ask first**
- Return all partitions? → Yes (exponential output).

> 💡 **Idea.** Cut off a palindrome prefix, recurse on the rest. Precompute which substrings are palindromes so each check is O(1).

**Brute force:** check palindromes from scratch at each cut, adds an O(n) factor.

**Steps**
1. At the end of the string, record the pieces.
2. For each end position, take `s[start:end]`.
3. If it is a palindrome, append it, recurse from `end`, pop.

**Cost:** Time O(n · 2ⁿ) · Space O(n)

**Likely follow-ups**
- **Minimum number of cuts (LC 132)?** DP: `cuts[i] = min(cuts[j − 1] + 1)` over palindromes `s[j..i]`; with the palindrome table it's O(n²).
- **Count the partitions?** DP: `ways[i] = sum(ways[j])` over palindromes `s[j..i-1]`.
- **Palindrome table?** `pal[i][j] = s[i] == s[j] and (j − i < 2 or pal[i+1][j−1])`.

In [48]:
def partition_palindromes(s):
    """Every palindromic partition of s."""
    out = []
    path = []

    def backtrack(start):
        if start == len(s):
            out.append(path.copy())
            return
        # try every next cut position
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            # only palindromic pieces may continue
            if piece == piece[::-1]:
                path.append(piece)
                backtrack(end)
                path.pop()

    backtrack(0)
    return out


assert partition_palindromes("aab") == [["a", "a", "b"], ["aa", "b"]]
assert partition_palindromes("a") == [["a"]]
print(partition_palindromes("aab"))

[['a', 'a', 'b'], ['aa', 'b']]


### 6.2.3 [93. Restore IP Addresses](https://leetcode.com/problems/restore-ip-addresses/) — ★ · Medium

**Problem.** All valid IPv4 addresses formed by inserting three dots into a digit string.  
**Example.** `"25525511135"` → `["255.255.11.135", "255.255.111.35"]`  
**Spot it.** Four segments, each 1–3 digits, value ≤ 255, no leading zeros.

**Ask first**
- Leading zeros allowed? → No — `"0"` is OK, `"01"` isn't.

> 💡 **Idea.** Place 3 dots: try segment lengths 1–3, keep a segment only if it's 0–255 with no leading zero. Need exactly 4 segments using all digits.

**Brute force:** try all 3-dot placements and validate — already small (at most 3⁴ = 81 tries).

**Steps**
1. Four parts and all digits used → record.
2. If the remaining digits cannot fit the remaining parts, return.
3. Try segment lengths 1–3; stop on leading zeros or a value > 255.
4. Recurse with the segment added.

**Cost:** Time O(3⁴) (at most 81 branches) · Space O(1) besides the output

**Likely follow-ups**
- **Validate an IP string (LC 468) — IPv4 or IPv6?** IPv4: 4 parts, 0–255, no leading zeros. IPv6: 8 groups of 1–4 hex digits.
- **Why is it effectively O(1)?** At most 3 choices × 4 segments; the string is at most 12 digits long.

In [49]:
def restore_ip_addresses(s):
    """All valid IPs from the digit string."""
    out = []

    def backtrack(start, parts):
        if len(parts) == 4:
            if start == len(s):
                out.append(".".join(parts))
            return
        # remaining digits must fit 1..3 per remaining part
        remaining = len(s) - start
        if not 4 - len(parts) <= remaining <= 3 * (4 - len(parts)):
            return
        for length in range(1, 4):
            seg = s[start : start + length]
            # too short, leading zero, or over 255: longer segments fail too
            if len(seg) < length or (seg[0] == "0" and length > 1) or int(seg) > 255:
                break
            backtrack(start + length, parts + [seg])

    backtrack(0, [])
    return out


assert restore_ip_addresses("25525511135") == ["255.255.11.135", "255.255.111.35"]
assert restore_ip_addresses("0000") == ["0.0.0.0"]
assert restore_ip_addresses("101023") == [
    "1.0.10.23",
    "1.0.102.3",
    "10.1.0.23",
    "10.10.2.3",
    "101.0.2.3",
]
print(restore_ip_addresses("25525511135"))

['255.255.11.135', '255.255.111.35']


### 6.2.4 [51. N-Queens](https://leetcode.com/problems/n-queens/) — ★★ · Hard

**Problem.** All placements of `n` non-attacking queens on an `n × n` board.  
**Example.** `n = 4` → 2 solutions, for example `[".Q..", "...Q", "Q...", "..Q."]`  
**Spot it.** One queen per row; track used columns and both diagonals in sets.

**Ask first**
- Return boards or just the count? → Boards here (LC 52 is the count).

> 💡 **Idea.** Place one queen per row. Track used columns and both diagonals (`r − c` and `r + c`) in sets so each check is O(1).

**Brute force:** try every placement of n queens, C(n², n).

**Steps**
1. Place queens row by row.
2. Skip column `c` if `c`, `r − c` or `r + c` is used.
3. Add to the three sets, recurse to the next row.
4. Remove from the sets; at row `n`, build the board.

**Cost:** Time O(n!) · Space O(n)

**Likely follow-ups**
- **Just the count (LC 52)?** Same search, count instead of building boards; bitmasks make it fast.

In [50]:
def solve_n_queens(n):
    """All N-Queens boards."""
    out = []
    # attacked columns, diagonals (r - c) and anti-diagonals (r + c)
    cols = set()
    diag = set()
    anti = set()
    queens = []

    def backtrack(r):
        if r == n:
            out.append(["." * c + "Q" + "." * (n - c - 1) for c in queens])
            return
        for c in range(n):
            if c in cols or r - c in diag or r + c in anti:
                continue
            # place the queen
            cols.add(c)
            diag.add(r - c)
            anti.add(r + c)
            queens.append(c)
            backtrack(r + 1)
            # remove the queen
            queens.pop()
            cols.remove(c)
            diag.remove(r - c)
            anti.remove(r + c)

    backtrack(0)
    return out


assert solve_n_queens(4) == [
    [".Q..", "...Q", "Q...", "..Q."],
    ["..Q.", "Q...", "...Q", ".Q.."],
]
assert len(solve_n_queens(1)) == 1
print(len(solve_n_queens(8)))

92


### 6.2.5 [698. Partition to K Equal Sum Subsets](https://leetcode.com/problems/partition-to-k-equal-sum-subsets/) — ★★ · Medium

**Problem.** Can the array be split into `k` subsets with equal sums?  
**Example.** `nums = [4, 3, 2, 3, 5, 2, 1], k = 4` → `True` (`[5]`, `[1,4]`, `[2,3]`, `[2,3]`)  
**Spot it.** Fill k buckets with backtracking; sort descending and skip symmetric states.

**Ask first**
- Is the total divisible by k? → If not, return False immediately.
- All positive? → Yes, so pruning on overflow is safe.

> 💡 **Idea.** Each bucket must sum to `total / k`. Sort big → small, then put each number into a bucket that still has room; skip buckets with the same current sum (they're interchangeable).

**Brute force:** try every assignment of numbers to buckets, kⁿ.

**Steps**
1. Sum must divide by k; the largest number must fit in `target`.
2. Sort descending (big items fail fast).
3. For each number, try each bucket with room; recurse; undo.
4. If the bucket was empty after undoing, stop trying other buckets.

**Cost:** Time O(kⁿ) worst case, heavily pruned · Space O(n + k)

**Likely follow-ups**
- **Bitmask DP / memo?** `dp[mask]` = can the used set `mask` be filled into complete buckets + a partial one? O(n·2ⁿ).
- **Why sort descending?** Big numbers fail fast, cutting the search tree early.
- **Why does greedy fail?** Filling one bucket greedily can block others, e.g. `[1,1,1,1,2,2,2,2]`, k = 4.

In [51]:
def can_partition_k_subsets(nums, k):
    """True if nums splits into k equal-sum groups."""
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    # largest first: failures show up early
    nums = sorted(nums, reverse=True)
    if nums[0] > target:
        return False
    buckets = [0] * k

    def backtrack(i):
        if i == len(nums):
            return True
        for b in range(k):
            if buckets[b] + nums[i] <= target:
                buckets[b] += nums[i]
                if backtrack(i + 1):
                    return True
                buckets[b] -= nums[i]
            # empty buckets are identical: one failure covers them all
            if buckets[b] == 0:
                break
        return False

    return backtrack(0)


assert can_partition_k_subsets([4, 3, 2, 3, 5, 2, 1], 4)
assert not can_partition_k_subsets([1, 2, 3, 4], 3)
print("ok")

ok


### 6.2.6 [473. Matchsticks to Square](https://leetcode.com/problems/matchsticks-to-square/) — ★★ · Medium

**Problem.** Use all matchsticks to form a square.  
**Example.** `[1, 1, 2, 2, 2]` → `True` (sides: `[2]`, `[2]`, `[2]`, `[1,1]`)  
**Spot it.** Partition into 4 equal-sum subsets.

**Ask first**
- Must every stick be used? → Yes, and sticks can't be broken.

> 💡 **Idea.** It's Partition to K Equal Sum Subsets with k = 4: each side must be `total / 4`. Sort descending and backtrack into the 4 sides.

**Brute force:** try every assignment of sticks to sides, 4ⁿ.

**Steps**
1. Fewer than 4 sticks → false.
2. Call the k-bucket partition with `k = 4`.

**Cost:** Time O(4ⁿ) worst case, pruned · Space O(n)

**Likely follow-ups**
- **Bitmask DP?** n ≤ 15, so `dp[mask]` = leftover length of the current side; O(n·2ⁿ).
- **k-sided shape?** Same code with k buckets (LC 698).
- **Best pruning?** Sort descending; skip a side if it has the same length as a side already tried; stop if any stick > side.

In [52]:
def makesquare(matchsticks):
    """True if the sticks form a square."""
    # four equal sides = four equal-sum buckets
    return len(matchsticks) >= 4 and can_partition_k_subsets(matchsticks, 4)


assert makesquare([1, 1, 2, 2, 2])
assert not makesquare([3, 3, 3, 3, 4])
print("ok")

ok


---
# Summary

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Grid DFS | islands, regions, flood fill | mark visited before recursing | O(mn) |
| Reverse search | border-connected, ocean flow | start from the target side and invert | O(mn) |
| Multi-source BFS | distance to nearest X, spreading | enqueue all sources at distance 0 | O(mn) |
| BFS on states | word ladder, lock | generate neighbours on the fly | O(states) |
| Topological sort | prerequisites, ordering, alien alphabet | Kahn: peel in-degree 0 | O(V + E) |
| Leaf peeling | tree centres | repeatedly remove degree-1 nodes | O(n) |
| Union–find | redundant edge, merging groups, equations | `find` with compression, union by size | ~O(1) |
| Dijkstra | non-negative weighted shortest path | min-heap, skip stale entries | O(E log V) |
| Bottleneck Dijkstra | minimise max edge on a path | `max` instead of `+` | O(E log V) |
| Bellman–Ford (k rounds) | at most k stops | relax all edges from the previous round's copy | O(kE) |
| MST (Prim) | connect all points cheaply | grow tree by cheapest outside node | O(n²) dense |
| Backtracking | all subsets / permutations / boards | choose → recurse → un-choose, prune early | exponential |